# W1D1｜PyTorch 张量操作 + 自动求导

> 学习日期：2026-04-10
> 目标：掌握 PyTorch 核心 API，理解自动求导机制，夯实 Day 1 基础

# 一、张量（Tensor）— 最底层数据结构

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| 创建方式 | `torch.tensor()` vs `torch.Tensor()` | 两者区别（是否拷贝数据、类型推断） |
| 数据类型 | `float/int/bool/dtype` | 如何指定 device + dtype 同时创建 |
| GPU 迁移 | `.cuda()` / `.to(device)` | 如何判断 GPU 可用 |
| 形状操作 | `view / reshape / transpose / permute` | view vs reshape 区别（是否连续） |
| 索引切片 | `tensor[mask]` / `torch.masked_select` | 如何按条件筛选 |
| 运算 | `matmul / mm / @` / `torch.sum / mean / max` | 矩阵乘法哪个最快/最安全 |
| 与 NumPy 互转 | `tensor.numpy()` / `torch.from_numpy(nparr)` | 共享内存问题 |

---

## 1.1 创建方式详解

## 四类方法参数对比

| 方法 | device 参数 | requires_grad | dtype 参数 | 数据来源 | 是否拷贝 |
|---|---|---|---|---|---|
| `torch.Tensor(data)` | ❌ 无 | ❌ 无 | ❌ 无（固定float32） | 传入已有数据 | ✅ 拷贝 |
| `torch.tensor(data)` | ✅ 有 | ✅ 有 | ✅ 有（自动推断） | 传入已有数据 | ✅ 拷贝 |
| `torch.from_numpy(arr)` | ❌ 无 | ❌ 无 | ❌ 无（跟随numpy） | 必须是numpy数组 | ❌ 共享内存 |
| `torch.zeros()` / `torch.ones()` | ✅ 有 | ✅ 有（默认False） | ✅ 有（默认float32） | 自己生成 | ✅ 独立 |

## torch.Tensor() — 类构造函数（不推荐）

实际上是 `torch.FloatTensor` 的别名，类型固定 float32，无其他参数。

In [ ]:
torch.Tensor([1, 2, 3])  # → [1., 2., 3.] float32
t = torch.Tensor([1., 2.]).to('cuda').requires_grad_(True)

## torch.tensor() — 工厂函数（推荐）

自动推断类型，拷贝数据，参数最全。

In [ ]:
torch.tensor([1, 2, 3])                            # int64
torch.tensor([1., 2., 3.])                         # float32
torch.tensor([1, 2], dtype=torch.float32)          # 强制 float32
x = torch.tensor([1., 2., 3.], device='cuda', requires_grad=True)  # 一步到位

## torch.from_numpy() — 共享内存（需注意坑）

必须传 numpy 数组，共享内存修改互相影响。

In [ ]:
import numpy as np
arr = np.array([1., 2., 3.])
t = torch.from_numpy(arr)  # 共享内存！
t[0] = 99.
print(arr)  # [99., 2., 3.] ← numpy 数组被改了！

dtype 跟随 numpy，不统一成 PyTorch 默认值。

## torch.zeros() / torch.ones() — 初始化

In [ ]:
torch.zeros(3, 4, dtype=torch.float32, device='cuda', requires_grad=True)
torch.ones(2, dtype=torch.int64)

## 默认值

所有方法不指定时 → **device=cpu，requires_grad=False**

---

## 1.2 形状操作

### stride 详解

stride 是"跳读规则"，决定按当前维度顺序读取 storage 时，每个维度内跳几个元素才能读完。

In [ ]:
x = torch.randn(2, 3, 4)  # shape: (2, 3, 4)
x.stride()  # → (12, 4, 1)

把 `(2, 3, 4)` 想象成 2 页纸，每页 3 行，每行 4 个元素。

### 连续 stride 的计算公式

In [ ]:
stride[i] = shape[i+1] × shape[i+2] × ... × shape[last]

## view

**本质**：换一套 shape/stride 来解释同一块 storage，数据完全不动。

In [ ]:
x = torch.randn(2, 3, 4)  # (2, 3, 4), stride (12, 4, 1)
x.view(6, 4)   # (6, 4), stride (4, 1) — 数据完全不动
x.view(24,)    # (24,)
x.view(1, 2, 3, 4)  # 任意维度都可以，只要 1×2×3×4 = 24
x.view(-1)     # (24,) — -1 自动推断

view 可以任意维度，不限于二维。要求：tensor 必须连续，不连续时报错：

In [ ]:
x_t = x.transpose(0, 1)  # stride (4, 12, 1) — 不连续
x_t.view(24)   # ❌ RuntimeError

## transpose

**本质**：交换两个维度的位置，数据不动，只换 shape 和 stride。

In [ ]:
x = torch.randn(2, 3)        # shape (2, 3), stride (3, 1)
x_t = x.transpose(0, 1)     # shape (3, 2), stride (1, 3)

**transpose 后 stride 的算法：直接交换对应位置的值。**

In [ ]:
x = torch.randn(2, 3, 4)           # stride (12, 4, 1)
x.transpose(0, 1)                   # stride (4, 12, 1)

transpose 后 tensor 必然不连续，因为 stride 顺序和 storage 物理顺序对不上了。

## reshape

**本质**：等价于 `contiguous().view()`。

In [ ]:
x.reshape(6, 4)  # 连续 tensor → 直接 view，不复制
x_t.reshape(6,)  # 不连续 tensor → 先复制重排，再 view

**优先用 `reshape`**，永远不报错。

## permute

入参语义：新维度 i 来自旧维度 dim_i。

In [ ]:
x = torch.randn(2, 3, 4)  # shape: (2, 3, 4)
x.permute(2, 0, 1)  # shape: (4, 2, 3)

和 transpose 完全一致——按同样索引直接重新排列 stride。

## squeeze / unsqueeze

In [ ]:
x = torch.randn(1, 3, 1, 8, 1)  # 5个维度
x.squeeze().shape   # (3, 8) — 删除所有 size=1 的维度
x.squeeze(0).shape  # (3, 1, 8, 1) — 删最外层
x.unsqueeze(0).shape   # (1, 2, 3) — 在最前面插

## flatten

In [ ]:
x = torch.randn(2, 3, 4)  # (2, 3, 4)
x.flatten().shape           # (24,) — 所有维度展平
x.flatten(1, 2).shape       # (2, 12) — 只展平中间两个维度

等价于 `reshape(-1)` 或 `reshape(..., -1)`。

## 三者对比

| | view | transpose | reshape |
|---|---|---|---|
| 数据动了吗？ | ❌ 没动 | ❌ 没动 | ✅ 不连续时会复制重排 |
| 连续性 | 要求连续 | 必然不连续 | 永远成功 |

---

## 1.3 索引切片

### 基础索引 + 切片

In [ ]:
x = torch.randn(2, 3, 4)
x[0].shape         # (3, 4) — 取第一个样本
x[0, 1].shape      # (4,) — 取特定元素
x[0:2].shape       # (2, 3, 4) — 切片
x[:, 1:].shape     # (2, 2, 4) — 跨维度切片

### None（等价于 unsqueeze）

In [ ]:
x = torch.randn(2, 3)
x[None, :].shape   # (1, 2, 3) — 等价于 x.unsqueeze(0)
x[:, None].shape   # (2, 1, 3)

### view vs copy 的分界线（核心！）

**返回 view**（共享底层 storage）：基础切片索引、`None` 索引、`squeeze`/`unsqueeze`、`transpose`/`permute`、`view`（连续时）、`expand`

**返回 copy**（独立新 storage）：`clone()`、布尔掩码索引、整数数组索引、`torch.masked_select`

In [ ]:
x = torch.randn(2, 3)
y = x[0:1]           # view
z = x[x > 0]          # copy
print(x.storage().data_ptr() == y.storage().data_ptr())  # True → view
print(x.storage().data_ptr() == z.storage().data_ptr())  # False → copy

### 布尔掩码

In [ ]:
x = torch.randn(3, 4)
mask = x > 0
x[mask].shape              # (?,) — 展平成一维
torch.masked_select(x, mask).shape  # 同上，完全等价

布尔掩码返回 **copy**，不共享数据。

### 整数数组索引

In [ ]:
x = torch.randn(5, 3)
rows = torch.tensor([0, 2, 3])
x[rows].shape        # (3, 3) — 取指定行
x[rows, 1].shape     # (3,) — 取这些行的第二列

整数数组索引返回 **copy**，不共享数据。

### 分步法：不连续行列组合

In [ ]:
x = torch.randn(5, 3)
rows = torch.tensor([0, 2, 3])
cols = torch.tensor([0, 2])
x[rows][:, cols].shape  # (3, 2) — 分两步取不连续行列

---

## 1.4 常用运算

### 逐元素运算

In [ ]:
x + 1      # 加减乘除基本运算
x ** 2     # 幂运算
torch.sqrt(x)  # 开方
x.abs().log()  # 链式调用

### 归约运算

In [ ]:
x.sum()              # 全部求和 → scalar
x.sum(dim=0)        # 按 dim=0 求和 → (3,)
x.mean()             # 均值
x.max()              # 最大值 → scalar
x.argmax()            # 最大值索引（flatten后）
x.max(dim=0)         # 返回 (values, indices)

### 矩阵运算

In [ ]:
a = torch.randn(3, 4)
b = torch.randn(4, 5)
(a @ b).shape   # (3, 5)

**推荐 `@` / `matmul`**，支持多维 + 广播，是全能版。

### 广播机制

从左补1，从右对齐检查。

In [ ]:
x = torch.randn(3, 4)
y = torch.randn(4)       # 1D
x + y  # y补成(1,4) → broadcast到(3,4) ✅

### torch.where

In [ ]:
torch.where(x > 0, x, 0)  # x>0保留x，否则填0（限幅）

### in-place 操作

下划线 `_` 结尾 = in-place，直接修改原 tensor。

In [ ]:
x.add_(1)           # x = x + 1，原地改
x.zero_()           # 全部变成 0

---

## 1.5 GPU 迁移与设备

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
x = torch.randn(10, device=device)
model = model.to(device)

---

## 1.6 与 NumPy 互转

In [ ]:
arr = x.numpy()              # Tensor → NumPy（共享内存时有问题）
x = torch.from_numpy(arr)    # NumPy → Tensor（共享内存）
x = tensor.numpy()           # 需要 requires_grad=False 或 detach

---

## 1.7 默认值总结

| 方法 | device | requires_grad | dtype |
|---|---|---|---|
| `torch.Tensor()` | cpu | False | float32 |
| `torch.tensor()` | cpu | False | 自动推断 |
| `torch.zeros/ones()` | cpu | False | float32 |
| `torch.from_numpy()` | cpu | False | 跟随numpy |

---

## 代码练习

In [ ]:
import torch
import numpy as np

# 1. torch.Tensor() vs torch.tensor() — 类型推断差异
t1 = torch.Tensor([1, 2, 3])       # 固定 float32，值变成 1.0, 2.0, 3.0
t2 = torch.tensor([1, 2, 3])       # 自动推断 int64

# 2. 拷贝验证
lst = [1., 2., 3.]
t = torch.Tensor(lst)
t[0] = 99.
print("原始 list 不变:", lst)  # 不受影响

# 3. from_numpy() 共享内存
arr = np.array([1., 2., 3.])
t_share = torch.from_numpy(arr)
t_share[0] = 99.
print("from_numpy 改了 numpy:", arr)  # [99., 2., 3.]

# 4. view vs reshape
x = torch.randn(2, 3)
x_t = x.transpose(0, 1)
try:
    x_t.view(6)
except RuntimeError as e:
    print("view 不连续报错:", e)
print("reshape 不连续:", x_t.reshape(6))

# 5. stride 验证
x = torch.randn(2, 3, 4)
print("原始 stride:", x.stride())  # (12, 4, 1)

# 6. 连续性判断
x = torch.randn(2, 3)
print("is_contiguous:", x.is_contiguous())
x_t = x.transpose(0, 1)
print("transpose 后 is_contiguous:", x_t.is_contiguous())

# 二、自动求导（Autograd）— PyTorch 灵魂

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| `requires_grad` | 默认为 False，设为 True 开启追踪 | 哪些操作会默认开启 |
| `backward()` | 反向传播计算梯度 | 何时调用，梯度会累加还是覆盖 |
| `grad` | 保存梯度值 | 多个 `backward()` 时梯度如何变化 |
| `grad_fn` | 记录创建张量的运算 | 用于什么 |
| `torch.no_grad()` | 前向推理时不追踪梯度 | 与 `eval()` 区别 |
| `detach()` | 截断计算图 | 何时需要 detach |
| `hook` 机制 | 注册前向/反向 hook | 有什么用 |

**⚠️ 面试必答题：**
- PyTorch 反向传播原理？计算图是怎么构建的？
- `backward()` 掉了梯度会怎样？多次 backward 梯度累加还是覆盖？
- 梯度消失/爆炸的原因？在 PyTorch 中如何检测和解决？

---

# 1-5 自动求导（Autograd）

## 1-5-1 计算图与反向传播原理

PyTorch 的自动求导基于**动态计算图**（Dynamic Computation Graph）。

**核心概念**：
- 每当对 `requires_grad=True` 的张量进行运算，PyTorch 会构建一个反向传播的 DAG
- 叶子节点（leaf nodes）是用户直接创建的张量（通过 `torch.tensor()` 等）
- 根节点（root）是最终输出标量（scalar），通常是一个 loss

---

In [ ]:
import torch

# 叶子节点：用户创建，requires_grad 默认为 False
x = torch.tensor([1., 2., 3.], requires_grad=True)  # 叶子
y = x ** 2          # 中间节点，grad_fn 记录运算
z = y.sum()         # 根节点，backward 从这里开始

print("x 是叶子:", x.is_leaf)
print("y 是叶子:", y.is_leaf)
print("z 是叶子:", z.is_leaf)
print("z.grad_fn:", z.grad_fn)  # <SumBackward0>

---

**DAG 的构建是动态的**：每次前向传播都会重新构建计算图，修改代码后图结构随之变化。

---

In [ ]:
# 同一个变量两次前向，建两个图
a = torch.tensor([1., 2.], requires_grad=True)
b = a ** 2
c = b * 2
d = c.sum()
print(d.grad_fn)  # <AddBackward1> — 两个图各自独立

---

### 动态图 vs 静态图

| | PyTorch（动态图） | TensorFlow 1.x（静态图） |
|--|---|---|
| 建图时机 | 前向传播时同步构建 | 先定义图，后执行 |
| 控制流 | 直接用 Python `if/for/while` | 需要 `tf.cond`/`tf.while_loop` |
| 调试 | 直接报错，可 print | 需要 `sess.run` 才能看值 |
| 灵活性 | 极高，代码怎么写图就怎么建 | 需预先定义所有分支 |

**动态图优势**：代码即模型，不用预先声明图结构。

**静态图优势**：图结构已知，可做全局优化（如算子融合、内存规划），性能更好。TensorFlow 2.x 默认 eager execution（动态图），但用 `tf.function` 可以 jit 编译成静态图加速。PyTorch 的 `torch.compile` 也是类似思路——先把动态图"冻结"成静态图再优化。

---

In [ ]:
# 动态图：每行代码立即执行
x = torch.tensor([1., 2.])
y = x * 2        # 立即计算
z = y + 1        # 立即计算

# 静态图（TensorFlow 风格）：
# x = tf.placeholder(tf.float32)   # 先声明
# y = x * 2                        # 只是画边，不计算
# with tf.Session() as sess:
#     result = sess.run(y, feed_dict={x: [1., 2.]})  # 实际执行

---

**Notebook 中的内存问题**：动态图下，每次 cell 跑前向都会创建新的计算图节点，如果不断保存中间结果（图节点被引用），内存会累积。定期 `del` 不需要的变量 + `gc.collect()` 可缓解。静态图因为图结构固定，不存在这个问题。

---

## 1-5-2 backward() 与梯度计算

`backward()` 从当前张量反向传播到所有叶子节点，计算 `∂tensor/∂leaf`。

**必须条件**：调用 `backward()` 的张量必须是**标量**（scalar，形状为空），或者传入 `gradient` 参数。

---

In [ ]:
import torch

# y = x²，求 dy/dx
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2       # y = [1, 4, 9]
z = y.sum()      # z = 14，必须是标量才能直接 backward()

z.backward()     # 等价于 z.backward(torch.tensor(1.))
print(x.grad)    # tensor([2., 4., 6.]) — dy/dx = 2x

# 验证：x=[1,2,3]，2x = [2,4,6] ✓

---

**非标量如何 backward**：需要传入 `gradient` 参数（与输出同形状），表示链式求导的起点。

---

In [ ]:
# J = [J₁, J₂]，每个 Jᵢ 对 x 求偏导
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2       # y = [1, 4, 9]
# y 有 3 个输出，每个都对 x 求偏导构成雅可比矩阵

v = torch.tensor([1., 0., 0.])  # 只想求 J₁ 对 x 的偏导
y.backward(v)
print(x.grad)     # tensor([2., 0., 0.]) = ∂J₁/∂x = 2x

---

---

## 1-5-2.5 标量 backward vs 向量 backward（gradient 参数详解）

### 核心问题：为什么非标量不能直接 backward？

标量 `loss.backward()` 直接能跑，是因为 PyTorch 知道"从 loss（一个数）往回传梯度，起点就是 1"。

但如果 `y` 是向量，`y.backward()` 会遇到歧义：**y 有多个元素，每个元素对上游参数的梯度都不一样**。这时候 PyTorch 不知道该把什么值传回去，所以直接报错：`grad can be implicitly created only for scalar outputs`。

### 用具体例子理解三种梯度

以一个完整的前向计算图为例：

```
x → [matmul] → y → [减法] → [平方] → [求平均] → loss
                                              ↑
                                         y_true
```

**已知数值**：
```
x = [1, 2]
w = [[1, 3], [2, 4]]
y_true = [10, 20]
```

**前向传播（逐步计算）**：

| 步骤 | 计算 | 结果 |
|------|------|------|
| y = x @ w | `y[0]=1×1+2×2, y[1]=1×3+2×4` | y = [5, 11] |
| diff = y - y_true | `5-10, 11-20` | diff = [-5, -9] |
| square = diff² | `(-5)², (-9)²` | square = [25, 81] |
| loss = mean(square) | `(25+81)/2` | loss = 53 |

**反向传播第一步：dloss/dy**

loss 对 y 求偏导——loss 是标量，y 是向量，所以 dloss/dy 也是向量：

```
loss = ((y[0]-10)² + (y[1]-20)²) / 2

∂loss/∂y[0] = 2×(y[0]-10) / 2 = y[0] - 10 = 5 - 10 = -5
∂loss/∂y[1] = 2×(y[1]-20) / 2 = y[1] - 20 = 11 - 20 = -9

所以 dloss/dy = [-5, -9]  ← 就是 diff 本身！
```

**反向传播第二步：dy/dw（雅可比矩阵）**

y 对 w 求偏导——y 是向量，w 是 2×2 矩阵，所以 dy/dw 是一个 2×2 的雅可比矩阵：

```
y[0] = x[0]×w[0,0] + x[1]×w[1,0] = 1×w[0,0] + 2×w[1,0]
y[1] = x[0]×w[0,1] + x[1]×w[1,1] = 1×w[0,1] + 2×w[1,1]

∂y[0]/∂w[0,0] = 1    ∂y[0]/∂w[0,1] = 0
∂y[0]/∂w[1,0] = 2    ∂y[0]/∂w[1,1] = 0
∂y[1]/∂w[0,0] = 0    ∂y[1]/∂w[0,1] = 1
∂y[1]/∂w[1,0] = 0    ∂y[1]/∂w[1,1] = 2

dy/dw = [[1, 0],   ← y[0] 对 w 的偏导
         [2, 0],
         [0, 1],   ← y[1] 对 w 的偏导
         [0, 2]]
```

注意：y[i] 只和 w[:, i]（第 i 列）有关，和 w[:, 1-i] 无关，所以每列只有一个元素有值。

**反向传播第三步：dloss/dw = dloss/dy · dy/dw（链式法则）**

链式法则公式：`∂loss/∂w[i,j] = ∂loss/∂y[j] × ∂y[j]/∂w[i,j]`

| | 第 0 列 (j=0) | 第 1 列 (j=1) |
|--|---------------|---------------|
| 第 0 行 (i=0) | `(-5) × 1 = -5` | `(-9) × 1 = -9` |
| 第 1 行 (i=1) | `(-5) × 2 = -10` | `(-9) × 2 = -18` |

```
dloss/dw = [[-5, -9],
             [-10, -18]]
```

### 标量 backward vs 向量 backward 的本质区别

| | `loss.backward()` | `y.backward(gradient)` |
|--|--|--|
| 起点梯度 | 自动假设 = 1（标量的上游梯度就是 1） | 你手动指定 upstream |
| 传播范围 | 从当前节点一直传到最后（叶子节点） | 只到当前节点为止 |
| 典型场景 | 训练神经网络算 loss 梯度 | 策略梯度、手动梯度构造 |

**`y.backward(gradient=[1,1])` 的实际含义**：

```
y → [matmul] → ... → loss
↑
手动传入 [1,1]，跳过 y 之后的所有计算
```

传入 `[1,1]` 相当于告诉 PyTorch："从 y 往上游传的时候，就传 [1,1] 就行了，不需要再算 loss → y 那一段的梯度了"。

也就是说：**把本应该从 loss 计算到 y 时产生的梯度（-5, -9），替换成你手动输入的值（1, 1）**。

---

In [ ]:
# 等价于：手动替换了 dloss/dy 本来的值 [-5, -9]
y.backward(gradient=torch.tensor([1.0, 1.0]))
# dloss/dw[j,k] = upstream[k] × x[j]
# 结果 = [[1, 1], [2, 2]]（而不是 [[-5,-9], [-10,-18]]）

---

---

## 1-5-3 梯度累加机制（关键！）

**默认行为：梯度是累加的，不是覆盖。**

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)

# 第一次 backward
y = (x ** 2).sum()
y.backward()
print("第1次:", x.grad)   # tensor([2., 4., 6.])

# 第二次 backward — 梯度会累加！
y2 = (x ** 2).sum()
y2.backward()
print("第2次:", x.grad)   # tensor([4., 8., 12.]) — 累加了！

# 正确的多步训练：每次前向之间清梯度
x.grad.zero_()           # 原地清零
y3 = (x ** 2).sum()
y3.backward()
print("清零后:", x.grad)   # tensor([2., 4., 6.]) — 正确

---

**训练循环中的标准写法**：

---

In [ ]:
for data, target in dataloader:
    optimizer.zero_grad()   # Step 1: 清梯度
    output = model(data)     # Step 2: 前向
    loss = criterion(output, target)
    loss.backward()          # Step 3: 反向
    optimizer.step()         # Step 4: 更新参数

---

**为什么设计成累加而不是覆盖？**

这是**有意为之**，而不是"忘了清零"。累加设计是为了支持两种场景：

**场景1：梯度累积（大 batch 训练）**

---

In [ ]:
# batch_size=64 显存不够，用4个 batch_size=16 累积
for i, (data, target) in enumerate(dataloader):
    output = model(data)
    loss = criterion(output, target) / 4  # 归一化
    loss.backward()   # 累加到 .grad
    
    if (i + 1) % 4 == 0:
        optimizer.step()      # 用累积的梯度更新
        optimizer.zero_grad()  # 4个batch累积完，清零

---

**场景2：多 loss 多路径回传**

---

In [ ]:
# 一个 shared 参数被多个分支使用
shared_params = ...
out1 = branch1(shared_params)
out2 = branch2(shared_params)

loss1 = criterion(out1, target1)
loss2 = criterion(out2, target2)

loss1.backward()  # shared_params.grad 有了第一份梯度
loss2.backward()  # 累加第二份，两个分支的梯度都被保留

---

**如果设计成"清零覆盖"**：上述两个场景都直接坏掉——要么无法做梯度累积，要么多路径的梯度互相覆盖。累加设计把控制权交给用户，框架不隐式做主。

---

In [ ]:
# 梯度累积：模拟 batch_size=64，实际用两个 batch_size=32
model.zero_grad()           # 清零
loss1 = model(batch1).sum() / 2  # 32
loss1.backward()             # 梯度 /2 累加
loss2 = model(batch2).sum() / 2  # 32
loss2.backward()             # 梯度 /2 累加
# 等效于 batch_size=64 的梯度

---

---

## 1-5-4 no_grad / set_grad_enabled / eval

### torch.no_grad()

**作用**：禁用梯度计算，不构建计算图，节省显存和计算量。

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2

with torch.no_grad():
    z = y * 2      # 这里不追踪梯度
    print("no_grad 中:", z.requires_grad)  # False

print("no_grad 外:", z.requires_grad)  # False — 不影响外面的变量

# 用于推理/评估阶段
model.eval()
with torch.no_grad():
    predictions = model(test_data)

---

**另一个写法**：`@torch.no_grad()` 装饰器，效果相同。

---

In [ ]:
@torch.no_grad()
def evaluate(model, data):
    return model(data)

---

### torch.set_grad_enabled()

**作用**：动态切换梯度追踪状态。

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)

torch.set_grad_enabled(False)
y1 = x * 2
print("关闭后:", y1.requires_grad)  # False

torch.set_grad_enabled(True)   # 重新开启
y2 = x * 2
print("开启后:", y2.requires_grad)  # True

---

### model.eval() vs model.train()

| 模式 | 作用 | 影响 |
|---|---|---|
| `train()` | 训练模式 | BatchNorm 用 batch 统计量，Dropout 生效 |
| `eval()` | 评估模式 | BatchNorm 用全局统计量（moving average），Dropout 不生效 |

---

In [ ]:
model.train()
for batch in train_loader:
    optimizer.zero_grad()
    loss = ...
    loss.backward()

model.eval()                    # 切换
with torch.no_grad():
    for batch in val_loader:
        ...

---

**注意**：`eval()` 不等于 `no_grad()`！两者针对不同问题，要叠加使用：

---

In [ ]:
model.eval()
with torch.no_grad():           # 双重保险
    val_loss = ...

---

---

## 1-5-5 detach() 截断计算图

**作用**：将张量从计算图中分离出来，创建一个**共享存储但不共享梯度追踪**的新张量。

---

In [ ]:
import torch

x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2                # y 仍在图中
z = y.detach()            # z 与 y 共享数据，但脱离梯度追踪

print("z.requires_grad:", z.requires_grad)  # False
print("y.requires_grad:", y.requires_grad)  # True

# z 可以正常计算，但不会影响 x 的梯度
z_new = z * 2             # 无梯度追踪

---

### 共享存储的陷阱

**detach() 出来的张量和原张量共享同一块底层内存**，修改其中一个会影响另一个：

---

In [ ]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x * 2      # y = [2, 4, 6]
z = y.detach() # z 和 y 共享底层存储

z[0] = 99      # 原地修改 z
print(y)        # y = [99, 4, 6]  ← y 也被改了！
print(z)        # z = [99, 4, 6]

---

**安全用法**：detach 后立即做只读操作（打印、存列表、画图），不会出问题。

**如果需要修改后不影响原值**，用 `clone()` 深拷贝：

---

In [ ]:
z = y.detach().clone()  # 深拷贝，不共享存储
z[0] = 99
print(y)  # y 不受影响

---

### 典型用途

1. **需要原地修改张量值时**（叶子节点不能直接 in-place 修改）
2. **把张量传给不需要梯度的函数**（如 numpy 操作、打印、画图）
3. **阻止梯度回传到某条分支**（如 Actor-Critic 中 detach baseline）

---

In [ ]:
# 错误示范：直接修改需要梯度的叶子节点
x = torch.tensor([1., 2., 3.], requires_grad=True)
x[0] = 10                   # RuntimeError: a view of a variable

# 正确做法
x_detached = x.detach()
x_detached[0] = 10          # OK

---

**detach() vs no_grad()**：
- `no_grad()`：**整个代码块**都不追踪梯度
- `detach()`：**单个张量**从图中分离出来（但共享存储）

---

## 1-5-6 hook 机制（进阶）

**作用**：在不需要修改前向/反向代码的情况下，**拦截**前向传播或反向传播的过程，查看或修改中间张量/梯度。

**注册层级说明**：hook 绑定的位置决定触发次数：

| 注册位置 | 触发次数 | 说明 |
|---------|---------|------|
| `model`（根模块） | 1次 | 拦截整个模型的最终输入输出 |
| `model.fc1`（子模块） | 1次 | 只在 fc1 算完时触发 |
| `model.fc2`（子模块） | 1次 | 只在 fc2 算完时触发 |

---

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 5),
    nn.Linear(5, 2)
)

# 注册在根模块：只触发1次（最终输出）
handle = model.register_forward_hook(
    lambda m, inp, out: print(f'根模块 hook，output shape: {out.shape}')
)

# 如果想拦截每个层，需要遍历注册
for name, module in model.named_children():
    module.register_forward_hook(
        lambda m, inp, out: print(f'  层 {name} hook')
    )

x = torch.randn(2, 10)
y = model(x)
# 输出：
#   层 0 hook
#   层 1 hook
#   根模块 hook

handle.remove()

---

### 注册反向 hook

---

In [ ]:
def backward_hook(module, grad_input, grad_output):
    print(f"输入梯度形状: {[g.shape for g in grad_input]}")
    print(f"输出梯度形状: {grad_output[0].shape}")
    return grad_input  # 可以修改输入梯度

handle = model.register_backward_hook(backward_hook)

x = torch.randn(2, 10, requires_grad=True)
y = model(x)
loss = y.sum()
loss.backward()

---

### 常用场景

1. **特征提取**：在不修改模型的情况下，获取中间层激活值
2. **梯度检查**：验证梯度计算是否正确
3. **梯度修改**：在反向传播时人为注入噪声、裁剪等

---

In [ ]:
# 完整例子：提取 ResNet 中间层特征
import torchvision.models as models

model = models.resnet18(pretrained=True)
features = {}

def hook_fn(name):
    def fn(module, input, output):
        features[name] = output.detach()
    return fn

model.layer1.register_forward_hook(hook_fn("layer1"))
model.layer2.register_forward_hook(hook_fn("layer2"))

---

---

## 1-5-7 梯度消失与爆炸

### 原因

多层链式求导时，梯度在反向传播中不断连乘：

- **梯度消失**：|∂L/∂W| < 1，连乘后趋近于 0 → 参数几乎不更新
- **梯度爆炸**：|∂L/∂W| > 1，连乘后趋近于 ∞ → 参数大幅震荡

### 核心机制详解

#### 梯度裁剪（直接修改梯度）

---

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

---

**原理**：计算所有参数梯度的 L2 范数 `||g||₂ = √(g₁² + g₂² + ...)`，如果超过 `max_norm`，等比例缩放回来。

---

In [ ]:
# 等价逻辑
total_norm = 0.0
for p in model.parameters():
    total_norm += p.grad.norm(2).item() ** 2
total_norm = total_norm ** 0.5

if total_norm > max_norm:
    clip_coef = max_norm / total_norm
    for p in model.parameters():
        p.grad.mul_(clip_coef)  # 直接缩放梯度

---

**特点**：这是**唯一一个直接在反向传播后修改梯度**的方法，属于"事后补救"。只能防止爆炸，不能防止消失。

#### 残差连接（加法保梯度）

```
普通层：output = H(x)
残差层：output = H(x) + x
```

**关键**：加法的反向传播是"分流"的——`∂(H+x)/∂x = ∂H/∂x + 1`。

无论 H(x) 的梯度多小，加个 `+1` 就让梯度永远不会消失。也不存在梯度爆炸的问题（加法不会让梯度相乘变大）。

#### 归一化（稳定激活值，间接稳定梯度）

BatchNorm 前向：

---

In [ ]:
mu = x.mean(dim=0)
var = x.var(dim=0)
x_norm = (x - mu) / sqrt(var + eps)
y = gamma * x_norm + beta

---

**对梯度的影响**：
- **间接**：把激活值压到稳定范围（前向数值稳定 → 反向梯度数值也稳定）
- **直接**：gamma/beta 作为可学习参数，提供稳定的梯度回传路径（形状固定为特征维度，不会随深度指数变化）

#### 激活函数（间接影响梯度）

激活函数在前向时改变激活值分布，间接影响梯度：

| 激活函数 | 前向特点 | 对梯度的影响 |
|---------|---------|-------------|
| Sigmoid | 输出 0~1 | 导数最大 0.25，深层连乘≈0，梯度消失 |
| Tanh | 输出 -1~1 | 导数最大 1，比 Sigmoid 好，但仍有消失问题 |
| ReLU | 负区=0，正区=原值 | 正区导数恒为 1，不消失 |

**ReLU 的梯度（工程定义）**：
```
x > 0: 梯度 = 1
x < 0: 梯度 = 0（截断）
x = 0: 梯度 = 0 或 1（工程规定，不是数学严格定义）
```

**注意**：ReLU 在 x=0 处数学上不可导，但工程上通过**规定 subgradient**（次梯度）让它能参与反向传播。

#### 权重初始化（预防）

权重太小 → 激活值逐层变小 → 梯度消失
权重太大 → 激活值逐层变大 → 梯度爆炸

合理的初始化（如 Kaiming/Xavier）让各层激活值和梯度的方差在合理范围，从源头降低消失/爆炸风险。

#### LSTM/GRU（门控机制）

RNN 循环使用同一个权重矩阵 W：`h_t = h_{t-1} @ W`

梯度回传时：`∂h_T/∂h_t = W^T @ W^T @ ... @ W^T`（T-t 次连乘）

LSTM/GRU 通过**门控**决定保留多少历史、多少新信息：
```
h_t = h_{t-1} * output_gate + new_gate * input_gate
```
梯度可以"抄近道"不经过所有 W 连乘，从根本上缓解消失/爆炸。

### 方法分类总结

| 方法 | 操作阶段 | 机制 |
|------|---------|------|
| 梯度裁剪 | **反向传播后** | 直接修改梯度，治标 |
| 残差连接 | 前向 | 加法+1保梯度，治本 |
| 归一化 | 前向 | 稳定激活值→稳定梯度，治本 |
| 激活函数 | 前向 | 改变激活值分布，间接影响 |
| 权重初始化 | 前向（训练前） | 预防，治本 |
| LSTM/GRU | 前向 | 门控减少连乘，治本 |

**大多数方法都是在"前向阶段预防"梯度问题，只有梯度裁剪是在"反向传播后直接修改梯度"。**

---

## 1-5-7.5 nn.Module 与 nn.Parameter

### 为什么需要 nn.Module？

纯 tensor 的问题：参数需要手动管理。

---

In [ ]:
# 纯 tensor 写法
W = torch.randn(10, 5, requires_grad=True)
b = torch.randn(5, requires_grad=True)
optimizer = torch.optim.SGD([W, b], lr=0.1)  # 手动传入参数列表

---

**nn.Module 的核心价值**：

1. **自动参数收集**：`model.parameters()` 把所有带梯度的参数收拢在一起
2. **设备统一管理**：`model.to('cuda')` 一次移动所有参数
3. **封装复用**：把参数和计算逻辑打包成独立模块

---

In [ ]:
# nn.Module 写法
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.randn(out_features))
    
    def forward(self, x):
        return x @ self.weight + self.bias

model = Linear(10, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # 自动收集所有参数

---

### nn.Parameter 是什么？

---

In [ ]:
nn.Parameter(tensor) ≈ torch.tensor(..., requires_grad=True) + 自动注册到父 Module

---

| | 普通 tensor | nn.Parameter |
|--|-----------|--------------|
| `requires_grad` | 默认 False | 默认 True |
| 被 `model.parameters()` 收集 | ❌ | ✅ |
| 能被 optimizer 更新 | ❌ | ✅ |

`nn.Parameter` 就是 `requires_grad=True` 的 tensor，外层包了一层"注册"逻辑。本质上还是个 tensor。

### 计算图视角下的 nn.Module

```
nn.Module
    ├── self.weight (nn.Parameter = requires_grad=True 的 tensor)
    ├── self.bias   (nn.Parameter = requires_grad=True 的 tensor)
    └── forward()
            ↓
        构建计算图（和纯 tensor 完全一样）
```

`nn.Module` 本身不参与计算图，它只是 Python 层的组织结构。真正参与计算图的是 `nn.Parameter`——它们本质上是 tensor。`model(x)` 执行时，实际上是在 tensor 层面做运算，构建计算图。**nn.Module 是组织的壳，nn.Parameter 才是参与梯度追踪的实际的 tensor。**

---

## 本节面试重点

**Q1: PyTorch 反向传播原理？计算图是怎么构建的？**
- PyTorch 构建动态有向无环图（DAG），`requires_grad=True` 的张量参与运算时自动记录
- `backward()` 从根节点反向遍历，叶子节点的 `.grad` 累加梯度

**Q2: `backward()` 多次调用，梯度累加还是覆盖？**
- 累加！每次 `backward()` 都会把新梯度加到 `.grad` 上
- 所以训练循环中必须 `optimizer.zero_grad()` 清零

**Q3: `no_grad()` vs `eval()` 区别？**
- `no_grad()`：不构建计算图，节省显存
- `eval()`：BN 用全局统计量，Dropout 不生效
- 推理时两者叠加：`model.eval() + with torch.no_grad()`

**Q4: `detach()` 什么时候用？**
- 需要对张量值做原地修改时（叶子节点不能直接修改）
- 需要把计算图中间结果传给不需要梯度的操作时

**Q5: 梯度消失/爆炸的解决？**
- 梯度裁剪（`clip_grad_norm_`）、残差连接、归一化、ReLU 替代 Sigmoid、权重初始化

---

## 1-5-8 梯度本质详解（数学直觉）

### 梯度是什么？

**梯度的物理含义：瞬时变化率（导数）在多维空间的推广。**

对于一元函数 `y = f(x)`，导数 `f'(x)` 描述的是：x 每增加 1 个单位，y 变化多少。

对于多元函数 `L = f(w₁, w₂, ..., wₙ)`，梯度 `∇L = [∂L/∂w₁, ∂L/∂w₂, ..., ∂L/∂wₙ]` 描述的是：**每个参数 wᵢ 每增加 1 个单位，loss L 变化多少。**

### 梯度为什么能用来更新参数？

**梯度指向的是函数值增大的方向。**

```
L(w) = (w - 5)²，最小值在 w=5
∇L = dL/dw = 2(w - 5)

w=3 时，∇L = -4（负的）
→ 负梯度方向 = 增大方向的反方向
→ w 应该往正方向走（增大）
→ w_new = 3 - lr×(-4) = 3 + lr×4
```

所以：**沿负梯度方向走，函数值下降。** 这就是梯度下降法的核心直觉。

### 链式法则是梯度的核心计算规则

链式法则把复杂函数的梯度拆解为基本运算的梯度组合：

```
L = sin(w²)
设 u = w²，则 L = sin(u)

dL/dw = dL/du · du/dw = cos(u) · 2w = 2w·cos(w²)
```

PyTorch 的 autograd 引擎就是把这个链式法则**自动化执行**：
- 前向传播：真正计算每个节点的数值
- 反向传播：沿着链式法则的路径，把梯度逐级回传

### 梯度的累加本质

梯度是**加性的**——复合函数的梯度等于各部分梯度的和。

```
L = f(g(h(x)))
dL/dx = dL/df · df/dg · dg/dh · dh/dx
```

每一级都在**乘**（链式），最后得到完整梯度。这就是为什么深层网络的梯度容易消失/爆炸——链路上连续乘了很多小于1或大于1的数。

### 为什么梯度是"瞬时"而非"平均"？

```
f(x) = x²，在 x=3 处
- 瞬时导数：f'(3) = 6（切线斜率）
- x从3到5的平均变化率：(25-9)/(5-3) = 8
```

梯度下降用的是瞬时斜率（6），不是平均斜率（8）。这意味着：
- 每一步都假设"当前切线能很好地近似曲线的一小段"
- 步子越小，线性近似的误差越小
- 这也是为什么学习率太大时会发散——步子大到线性近似已经严重失真

### 梯度为负时更新的方向

```
w_new = w - lr × gradient
```

| 梯度方向 | 含义 | 更新动作 | 结果 |
|---|---|---|---|
| 负梯度 | w 增加则 L 减少 | w 增大 | L 减少 ✅ |
| 正梯度 | w 增加则 L 增加 | w 减少 | L 减少 ✅ |

负梯度方向 = 函数值下降最快的方向，这就是**最速下降法**（Steepest Descent）。

---

## 1-5-9 一阶 vs 二阶优化：工业界的现实选择

### 一阶方法的本质

一阶方法只用**一阶导数（梯度）** 来更新参数：

```
w_new = w - lr × ∇L(w)
```

**优点**：
- 计算量小：只需一次前向 + 一次反向，O(n)
- 存储少：只需存梯度向量，O(n)

**缺点**：
- 用线性近似对付非线性曲面，需要多步迭代
- 无法利用曲率信息，不知道步子该迈多大

### 二阶方法的本质

二阶方法利用**二阶导数（Hessian 矩阵）** 来更新参数：

```
w_new = w - H⁻¹ · ∇L(w)
```

其中 `H[i,j] = ∂²L/∂wᵢ∂wⱼ` 是所有二阶偏导数构成的矩阵。

**优点**：
- 对二次函数一步到位（理论最优）
- Hessian 包含曲率信息，步长自然确定，无需学习率

**缺点**：
- Hessian 是 n×n 矩阵，存储 O(n²)
- 计算 Hessian 或其逆是 O(n²) 或更高
- 对于 70B 参数的模型，Hessian 完全无法存储和计算

### 为什么工业界选一阶？

| 维度 | 一阶 | 二阶（Hessian） |
|---|---|---|
| 存储 | O(n) | O(n²) |
| 计算量 | O(n) | O(n²) ~ O(n³) |
| 70B 参数存储 | ~280GB | ~49万亿 GB |
| 可行性 | ✅ | ❌ |

### 工业界的工程 tricks

虽然二阶不可行，但工程 tricks 通过**近似曲率信息**来弥补：

#### 1. 自适应学习率（Adam/RMSProp）

**SGD 的问题**：所有参数用同一个学习率，但不同参数的梯度大小可能差几十倍。

**Adam 解决思路**：让梯度大的时候步子小，梯度小的时候步子大。

---

In [ ]:
# Adam 更新公式
m = β₁·m + (1-β₁)·∇L      # 梯度的一阶矩估计（类似 momentum）
v = β₂·v + (1-β₂)·∇L²     # 梯度平方的 EMA
w = w - lr·m / (√v + ε)

---

| 参数 | 含义 | 初始值 | 固定/可变 |
|------|------|--------|----------|
| `β₁ = 0.9` | m 的衰减率 | 固定 | 超参数，可调 |
| `β₂ = 0.999` | v 的衰减率 | 固定 | 超参数，可调 |
| `lr` | 学习率 | 固定 | 人调，不学习 |
| `ε = 1e-8` | 防止除零 | 固定 | 工程参数 |
| `m` | 梯度 EMA | `0` | 每步更新 |
| `v` | 梯度平方 EMA | `0` | 每步更新 |

**核心机制**：分母 `√v` 自适应调整学习率——梯度大的参数 `√v` 也大，学习率被压小；梯度小的参数 `√v` 也小，学习率被放大。

**与二阶方法的关系**：Newton 法 `w_new = w - H⁻¹·∇L` 用 Hessian 提供最优步长，但 Hessian 存储 O(n²) 不可行。Adam 用 `v` 估计 `E[∇L²]`（Hessian 对角线），用 `m` 估计 `∇L`，本质是**用一阶计算量换二阶效果的部分近似**。这就是 Adam（Adaptive Moment Estimation）名字的由来。

---

In [ ]:
# 直观理解：不同参数自动获得不同学习率
if 梯度大: lr被压小  # 防止震荡
if 梯度小: lr被放大  # 加速收敛

---

RMSProp 和 Adam 思路类似，区别在于 Adam 多了一个 `m`（momentum）来平滑更新方向。

#### 2. 学习率衰减 / 余弦退火

```
lr(t) = lr_max × cos(t/T_max × π)
```

本质：初期大步探索，后期小步收敛。越接近最优点，曲面越接近二次函数，小步长的线性近似更准确。

#### 3. Warmup

```
前N步：lr 从小慢慢增大
之后：正常衰减
```

防止早期曲率估计不稳定时步子太大。

#### 4. 梯度裁剪

---

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

---

防止梯度爆炸，本质是给步长加了一个安全上界。

### 理论的尴尬现状

| 方法 | 理论证明 | 实际情况 |
|---|---|---|
| 梯度下降（凸） | 严格收敛 | 非凸下不保证 |
| Adam | 有收敛证明 | 假设与实际场景不符 |
| AdamW | 无 | 经验性，工业界大量使用 |
| Warmup | 无 | 经验性，效果稳定 |

**一句话总结**：深度学习优化器是"工程经验驱动，理论严重滞后"的领域。一阶方法是当前大规模训练的唯一可行解，工程 tricks 本质上都是在用近似曲率信息弥补二阶信息的缺失。

---

## 1-5-10 PyTorch autograd 核心 API 速查

### 计算图与梯度追踪

| API | 作用 | 典型用法 |
|---|---|---|
| `requires_grad=True` | 开启当前张量的梯度追踪 | `x = torch.tensor([1.], requires_grad=True)` |
| `x.requires_grad` | 查看是否追踪梯度 | 条件判断 |
| `x.is_leaf` | 是否叶子节点（用户创建） | 调试时检查 |
| `x.grad_fn` | 反向传播函数引用 | 调试：`print(y.grad_fn)` |
| `x.grad` | 存储累积的梯度值 | 反向后：`print(x.grad)` |

### 反向传播

| API | 作用 | 典型用法 |
|---|---|---|
| `loss.backward()` | 标量反向传播 | `loss.backward()` |
| `loss.backward(gradient)` | 非标量反向传播 | `y.backward(grad_y)` |
| `retain_graph=True` | 保留计算图供二次反向 | `loss.backward(retain_graph=True)` |
| `create_graph=True` | 在 grad 中构建计算图 | 用于求高阶导 |

---

In [ ]:
# 求二阶导示例
x = torch.tensor([2.], requires_grad=True)
y = x ** 3
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]
d2y_dx2 = torch.autograd.grad(dy_dx, x)[0]  # 二阶导 = 6x = 12

---

### 梯度控制

| API | 作用 | 典型用法 |
|---|---|---|
| `optimizer.zero_grad()` | 清零所有参数的梯度 | 训练循环必备 |
| `x.grad.zero_()` | 清零特定张量梯度 | 原位操作 |
| `with torch.no_grad():` | 禁用梯度计算 | 推理时 |
| `@torch.no_grad()` | 装饰器版本 | 函数定义 |
| `torch.set_grad_enabled(False)` | 动态开关 | 条件分支推理 |
| `x.detach()` | 分离张量，截断计算图 | 需要非梯度操作时 |
| `x.detach_()` | 原地分离 | 少用 |

### Hook 机制

| API | 作用 | 典型用法 |
|---|---|---|
| `register_forward_hook(fn)` | 拦截前向传播 | 特征提取 |
| `register_backward_hook(fn)` | 拦截反向传播 | 梯度检查/修改 |
| `register_hook(fn)` | 给 grad 注册 hook | 打印或修改梯度 |
| `handle.remove()` | 取消注册 | 防止内存泄漏 |

---

In [ ]:
# 给梯度注册 hook（反向传播时触发）
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2
loss = y.sum()

def print_grad(grad):
    print("梯度:", grad)

y.register_hook(print_grad)
loss.backward()  # 打印：梯度: tensor([2., 4., 6.])

---

### 优化器中的梯度

---

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 完整训练循环
for data, target in dataloader:
    optimizer.zero_grad()        # Step 1: 清梯度
    output = model(data)          # Step 2: 前向（计算图自动构建）
    loss = criterion(output, target)
    loss.backward()              # Step 3: 反向（梯度存到 parameter.grad）
    optimizer.step()             # Step 4: 用梯度更新参数

---

### 梯度检查（Gradient Checking）

用于验证 autograd 计算的梯度是否正确（数值法近似 vs 解析法）：

---

In [ ]:
def gradient_check(model, x, y, eps=1e-5):
    model.eval()
    x.requires_grad = True
    output = model(x)
    loss = criterion(output, y)
    loss.backward()

    # 数值梯度
    for p in model.parameters():
        if p.grad is not None:
            numerical = []
            analytical = p.grad.data.clone()
            for i in range(min(5, p.numel())):  # 只检查前5个
                old = p.data.view(-1)[i].item()
                p.data.view(-1)[i] = old + eps
                loss_plus = criterion(model(x), y).item()
                p.data.view(-1)[i] = old - eps
                loss_minus = criterion(model(x), y).item()
                numerical.append((loss_plus - loss_minus) / (2 * eps))
                p.data.view(-1)[i] = old
            print(f"数值: {numerical[:3]}, 解析: {analytical.view(-1)[:3]}")

---

## 三、`nn.Module` — 模型构建核心

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| 继承 `nn.Module` | 必须重写 `__init__` + `forward` | 为什么要继承 |
| `super().__init__()` | 调用父类构造函数 | 不调用会怎样 |
| `named_parameters()` / `parameters()` | 遍历模型参数 | 如何冻结部分层 |
| `state_dict()` / `load_state_dict()` | 模型序列化/加载 | 怎么只加载部分参数 |
| `children()` / `modules()` | 遍历子模块 | 区别是什么 |
| 常见层 | `Linear / Conv2d / BatchNorm / Dropout / LSTM / Embedding` | 参数含义 |
| Winograd | 卷积计算优化算法 | F(m,r) 通用形式 |

**⚠️ 面试必答题：**
- `nn.Module` 的 `forward` 为什么只需写前向，反向自动搞定？
- `model(img)` 背后发生了什么？（call → forward → hooks）
- `model.train()` vs `model.eval()` 区别？（BN 和 Dropout 的行为差异）

# 三、nn.Module — 模型构建核心

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| 继承 `nn.Module` | 必须重写 `__init__` + `forward` | 为什么要继承 |
| `super().__init__()` | 调用父类构造函数 | 不调用会怎样 |
| `named_parameters()` / `parameters()` | 遍历模型参数 | 如何冻结部分层 |
| `state_dict()` / `load_state_dict()` | 模型序列化/加载 | 怎么只加载部分参数 |
| `children()` / `modules()` | 遍历子模块 | 区别是什么 |
| 常见层 | `Linear / Conv2d / BatchNorm / Dropout / LSTM / Embedding` | 参数含义 |

**⚠️ 面试必答题：**
- `nn.Module` 的 `forward` 为什么只需写前向，反向自动搞定？
- `model(img)` 背后发生了什么？（call → forward → hooks）
- `model.train()` vs `model.eval()` 区别？（BN 和 Dropout 的行为差异）

---

# 3. nn.Module — 模型构建核心

## 3-1 模块定义与参数管理

### 为什么要继承 nn.Module？

`nn.Module` 是 PyTorch 的模型组织框架。继承它不是为了"继承方法"，而是为了获得它提供的**基础设施**：

1. **自动参数收集** — `model.parameters()` 把所有 `nn.Parameter` 收拢，optimizer 一行搞定
2. **设备统一管理** — `model.to('cuda')` 一次把所有参数和数据搬到 GPU
3. **状态追踪** — 前向/反向 hooks、buffer 管理、模型结构序列化

---

In [ ]:
import torch
import torch.nn as nn

# 不继承 nn.Module：纯手工，每个 tensor 手动管理
W = torch.randn(10, 5, requires_grad=True)
b = torch.randn(5, requires_grad=True)
optimizer = torch.optim.SGD([W, b], lr=0.1)  # 手动传参

# 继承 nn.Module：自动化的基础设施
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()                          # ← 必须调用
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.randn(out_features))

    def forward(self, x):
        return x @ self.weight + self.bias

model = Linear(10, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # 一行搞定所有参数

---

### super().__init__() 为什么必须调用？

`nn.Module.__init__()` 里注册了若干 PyTorch 内部簿记结构——参数注册表、buffer 注册表、forward hook 容器等。不调用 super，`self.weight` 就不会被当成参数收集，`model.parameters()` 里就没有它，`model.to('cuda')` 也不会移动它。

---

In [ ]:
class BadLinear(nn.Module):
    def __init__(self, in_f, out_f):
        # 忘记调用 super().__init__()
        self.weight = nn.Parameter(torch.randn(in_f, out_f))  # 不会被注册！

model = BadLinear(10, 5)
print("parameters:", list(model.parameters()))  # [] — 空！weight 丢失了
print("weight 是否在 params:", model.weight in model.parameters())  # False

---

### nn.Parameter 是什么？

---

In [ ]:
nn.Parameter(tensor) ≈ requires_grad=True + 自动注册到父 Module

---

| | 普通 `torch.tensor` | `nn.Parameter` |
|--|---|---|
| `requires_grad` | 默认 False | 默认 True |
| 被 `parameters()` 收集 | ❌ | ✅ |
| 能被 optimizer 更新 | ❌ | ✅ |
| 本质 | 就是 tensor | 就是 tensor（外面包了注册逻辑） |

`nn.Parameter` 本质上就是个 `requires_grad=True` 的 tensor，只是外层套了一层"把自己注册到父 Module"的逻辑。没有其他魔法。

### 计算图视角下的 nn.Module

```
nn.Module (Python 对象)
    ├── self.weight (nn.Parameter = requires_grad=True 的 tensor)
    ├── self.bias   (nn.Parameter = requires_grad=True 的 tensor)
    └── forward()
            ↓
        tensor 运算 → 构建计算图（和纯 tensor 完全一样）
```

**关键**：nn.Module 本身不参与计算图，它只是 Python 层的组织壳。真正在计算图里的是 `nn.Parameter`（tensor）。`model(x)` 执行时，实际上是在 tensor 层面做运算，构建计算图。

---

In [ ]:
# 验证：Module 本身不在计算图里
x = torch.tensor([1., 2.], requires_grad=True)
linear = nn.Linear(2, 3)
y = linear(x)  # linear 是个 Python 对象，不是 tensor

print("linear.weight 在图中:", linear.weight.requires_grad)  # True
print("linear.weight.grad_fn:", linear.weight.grad_fn)      # None — 叶子节点！
print("y.grad_fn:", y.grad_fn)                             # <AddmmBackward>

---

### forward 为什么只需写前向？

因为 PyTorch 的 autograd 引擎根据**前向运算的每一步**，自动构建反向传播图。`forward` 里写了什么运算，autograd 就自动生成对应的反向函数（grad_fn）。

`model(x)` 背后发生了什么：

---

In [ ]:
output = model(x)

# 等价于：
output = nn.Module.__call__(model, x)
# 1. model.__call__() 执行（Python 魔法方法）
# 2. 调用 model.forward(x)                    ← 你的代码在这里
# 3. 执行所有注册的 forward_hooks            ← 可选拦截点
# 4. 返回 output

---

`__call__` 是 Python 的语法糖：任何 `obj(args)` 调用，实际是 `obj.__call__(args)`。PyTorch 在 `nn.Module.__call__` 里插入了一些钩子（forward pre-hook、forward post-hook），所以直接写 `forward` 不够——需要经过 `__call__`。这也是为什么 `model(x)` 和 `model.forward(x)` **不完全等价**。

---

## 3-2 遍历与结构（children / modules / named_parameters）

### 四个遍历方法的区别

---

In [ ]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(10, 8),
    nn.ReLU(),
    nn.Sequential(
        nn.Linear(8, 4),
        nn.ReLU()
    )
)

---

| 方法 | 返回内容 | 层级 |
|------|---------|------|
| `model.modules()` | 所有模块（包括自己、根） | 深度优先遍历（根 → 子 → 子...） |
| `model.children()` | 直接子模块（不包括嵌套更深） | 仅一层 |
| `model.named_parameters()` | 所有参数的 (name, tensor) | 深度遍历 |
| `model.parameters()` | 所有参数的 tensor | 深度遍历 |

---

In [ ]:
print("=== modules() (所有模块，包括嵌套) ===")
for m in model.modules():
    print(f"  {m}")

print("\n=== children() (仅直接子模块) ===")
for m in model.children():
    print(f"  {m}")

print("\n=== named_parameters() ===")
for name, p in model.named_parameters():
    print(f"  {name}: {p.shape}")

---

输出：
```
=== modules() ===
  Sequential(
    (0): Linear(in_features=10, out_features=8)
    (1): ReLU()
    (2): Sequential(
      (0): Linear(in_features=8, out_features=4)
      (1): ReLU()
    )
  )
  Linear(in_features=10, out_features=8)
  ReLU()
  Sequential(...)
  Linear(in_features=8, out_features=4)
  ReLU()

=== children() ===
  Linear(in_features=10, out_features=8)
  ReLU()
  Sequential(...)

=== named_parameters() ===
  0.weight: torch.Size([8, 10])
  0.bias: torch.Size([8])
  2.0.weight: torch.Size([4, 8])
  2.0.bias: torch.Size([4])
```

### 实际应用：冻结部分层

---

In [ ]:
# 冻结所有 children 中名字包含 'bias' 的参数
for name, param in model.named_parameters():
    if 'bias' in name:
        param.requires_grad = False

# 或者用 children 遍历，子模块是顺序存储的
for i, child in enumerate(model.children()):
    if i < 2:  # 冻结前两层
        for param in child.parameters():
            param.requires_grad = False

# 验证
trainable = [p for p in model.parameters() if p.requires_grad]
print("可训练参数:", sum(p.numel() for p in trainable))

---

### 冻结参数在梯度链中的行为

**冻结参数（`requires_grad=False`）不等于截断梯度**。冻结参数的梯度原封不动继续往上传，冻结只是"不记录该参数的梯度"。

---

In [ ]:
import torch

x = torch.tensor([1., 2.], requires_grad=True)
w_frozen = torch.tensor([[1., 0.], [0., 1.]], requires_grad=False)  # 冻结
w_trainable = torch.tensor([[2., 0.], [0., 2.]], requires_grad=True)  # 可训练

y = x @ w_frozen @ w_trainable
loss = y.sum()
loss.backward()

print("x.grad:", x.grad)              # 有梯度（w_frozen 没阻断）
print("w_trainable.grad:", w_trainable.grad)  # 有梯度

---

梯度路径：`loss → w_trainable → w_frozen → x`

w_frozen 被跳过（不记录梯度），但梯度继续传给它上游的 x。

**冻结 vs detach 的本质区别**：

| 操作 | 梯度流过？ | 记录该参数梯度？ |
|------|-----------|----------------|
| `requires_grad=False` | ✅ 畅通，梯度原封不动 | ❌ 不记录 |
| `detach()` | ❌ **截断**，梯度停止 | — |

**冻结参数的典型应用**：预训练模型 backbone 冻住，只 fine-tune 头部。backbone 的权重作为已知的"常量"，只更新 head 的参数——backbone 本身不更新，但梯度仍然流过它。

---

## 3-3 state_dict 与模型保存加载

### 两种保存方式的区别

---

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 2)

# 方式一：保存整个模型（不推荐）
torch.save(model, '/tmp/model_full.pth')  # 包含类定义，重载依赖类
loaded1 = torch.load('/tmp/model_full.pth')

# 方式二：保存 state_dict（推荐）
torch.save(model.state_dict(), '/tmp/model_sd.pth')  # 只有参数字典
loaded2 = nn.Linear(10, 2)
loaded2.load_state_dict(torch.load('/tmp/model_sd.pth'))  # 需要先实例化

---

**为什么推荐 state_dict**：
- 文件更小（只存参数，不存模型结构）
- 不依赖类定义，迁移性更强
- 断点续训时通常需要同时保存 optimizer 的 state_dict

### 完整断点续训

---

In [ ]:
# 保存
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.234,
}
torch.save(checkpoint, '/tmp/ckpt.pth')

# 加载
ckpt = torch.load('/tmp/ckpt.pth')
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt['epoch'] + 1

---

### 部分加载（strict / 非 strict）

---

In [ ]:
# 场景：预训练模型有 100 个参数，但你的模型改了最后层，只有 80 个

# strict=False：忽略不匹配的 key
pretrained = {'layer1.weight': ..., 'layer1.bias': ..., 'layer2.weight': ...}
model = MyModel()  # 只有 layer1
model.load_state_dict(pretrained, strict=False)  # layer2 缺失被忽略

# 精确过滤：只加载部分参数
pretrained = torch.load('/tmp/pretrained.pth')
partial = {k: v for k, v in pretrained.items() if 'fc' not in k}
model = MyModel()
model.load_state_dict(partial, strict=False)

---

---

## 3-4 train / eval 模式与 BatchNorm / Dropout 机制

### Dropout 机制

---

In [ ]:
import torch
import torch.nn as nn

dropout = nn.Dropout(p=0.5)  # 训练时随机丢弃 50%

x = torch.ones(5)
print("train 模式:", dropout(x))   # 训练时有随机性，部分元素变 0
print("eval 模式:", dropout.eval()(x))  # eval 时所有元素保留原值

# 注意：eval 模式需要用 dropout.eval()，而不是 dropout(x) 在 eval 之后调用
dropout.train()   # 切换到训练模式
dropout.eval()    # 切换到评估模式

---

**Dropout 训练 vs 评估的行为差异**：

| 模式 | 行为 | 效果 |
|------|------|------|
| train | 随机置零（p 的概率），剩余元素除以 `(1-p)` 做缩放 | 防止过拟合 |
| eval | 所有元素原样通过 | 确定性输出 |

**缩放的必要性**：`E[dropout(x)] = (1-p) × x/(1-p) = x`，数学期望不变，训练和测试的输出期望一致。

### BatchNorm 机制

---

In [ ]:
import torch
import torch.nn as nn

bn = nn.BatchNorm1d(3)  # 3 个特征

x = torch.randn(4, 3)  # batch=4, features=3
print("train 模式:")
out_train = bn(x)
print("  输出:\n", out_train)
print("  mean per feature:", out_train.mean(dim=0))  # ≈ [0, 0, 0]
print("  var per feature:", out_train.var(dim=0))   # ≈ [1, 1, 1]

print("\neval 模式:")
out_eval = bn.eval()(x)
print("  输出:\n", out_eval)  # 和 train 完全不同！

---

**BatchNorm 训练 vs 评估的行为差异**：

| 模式 | 计算均值/方差 | 使用均值/方差 |
|------|-------------|-------------|
| train | 用当前 batch 的统计量 | 当前 batch 的统计量 |
| eval | 不计算 | 用**全局移动平均**（moving average，训练时累积的） |

---

In [ ]:
# 验证：eval 用的 moving average 是训练时累积的
bn = nn.BatchNorm1d(3)
bn.train()

print("training 模式:")
for i in range(3):
    x = torch.randn(4, 3) * (i + 1)  # 越来越大的方差
    out = bn(x)
    print(f"  batch {i} mean: {out.mean(dim=0)}")

print("\n切换 eval 后（使用 moving average）:")
bn.eval()
x = torch.randn(4, 3)
print("  输出:", bn(x))
print("  running_mean (不更新):", bn.running_mean)
print("  running_var (不更新):", bn.running_var)

---

### model.train() vs model.eval() 实际影响

---

In [ ]:
model = nn.Sequential(
    nn.Linear(10, 8),
    nn.BatchNorm1d(8),
    nn.ReLU(),
    nn.Dropout(0.3)
)

model.train()
print("train 模式 - BatchNorm 用 batch 统计:")
x = torch.randn(8, 10)
print("  BatchNorm running_mean:", model[1].running_mean[:3])

model.eval()
print("\neval 模式 - BatchNorm 用 moving average:")
print("  BatchNorm running_mean:", model[1].running_mean[:3])

---

**总结：哪些层受 train/eval 影响？**

| 层 | train 行为 | eval 行为 |
|---|---|---|
| BatchNorm | 用 batch 统计量 | 用 moving average |
| Dropout | 随机置零 | 全部通过 |
| LayerNorm | 用 batch 统计量 | 用 batch 统计量（不变化） |
| InstanceNorm | 用 batch 统计量 | 用 running average |

---

### 补充：Dropout 的反向传播机制

Dropout 的前向过程是 `out = x * mask / (1-p)`，其中 mask 是随机生成的 0/1 掩码。

**反向传播时，mask 本身不参与梯度计算**（没有梯度），所以：

```
上游梯度 × mask / (1-p)
→ 被 mask=0 置零的位置，梯度 = 0（该神经元这轮不更新）
→ 被 mask=1 保留的位置，梯度正常回传
```

**关键**：Dropout 不阻断梯度流，只是"杀死"被 mask 置零的那些神经元，让它们这轮不更新。下一轮前向时重新生成 mask，神经元可能又活了。

**极端 Bottleneck 场景**：如果网络的窄层只有一个参数连接到前后两部分，该参数被 mask=0 置零时，前后两部分的梯度都会变成 0——这轮训练完全失效。实际网络有冗余路径，所以不严重。

### 补充：Norm 家族深度解析 — CV vs NLP 的本质差异

#### 三种 Norm 的归一化维度对比

| Norm | 归一化维度 | 归一化时参考的样本/位置 | 典型场景 |
|------|-----------|----------------------|---------|
| **BatchNorm** | 每个通道，跨 batch | 同一通道的所有样本同一位置 | 图像分类（CNN） |
| **LayerNorm** | 每个样本，跨所有维度 | 每个样本内部 | Transformer / NLP |
| **InstanceNorm** | 每个样本的每个通道 | 每个样本 × 每通道的空间区域 | 风格迁移 |

#### 核心直觉：在哪个维度做 Norm，就是消除哪个维度的差异

| Norm | 消除的差异 | 保留的差异 |
|------|-----------|-----------|
| **BatchNorm** | 不同样本在同一通道上的绝对水平差异 | 同一样本内部通道间的相对关系 |
| **LayerNorm** | 每个样本内部向量各维度的绝对强度 | 向量在高维空间里的**方向**（语义主要在方向里） |
| **InstanceNorm** | 每个样本每个通道的全局统计量 | 内容结构，去掉风格纹理 |

#### BatchNorm 为什么对图像有效，对 NLP 无效？

**图像**：不同图片在同一个空间位置 (h, w) 具有相同的**视觉语义含义**。所有图片在 (5, 7) 位置都是"某个局部纹理或边缘"，通道 C 在该位置的激活值跨图片具有相似的统计分布——BatchNorm 的假设成立。

**NLP**：第 0 个词和第 0 个词之间没有任何语义对齐。"The" 和 "A" 都出现在句子开头，但含义完全无关。跨样本在同位置做归一化，强行把不同语义的 token 拉成相同分布，破坏了语义信息。

---

In [ ]:
# 两个句子，同一位置 pos=0
# 句子A："The cat sits"     → token[0] = "The"
# 句子B："A dog runs"       → token[0] = "A"
# BatchNorm 强制让两个位置的激活值统计分布相同——灾难性的

---

#### LayerNorm 保留的是什么？

LayerNorm 对每个 token 的向量做归一化，数学上等价于将向量投影到**高维球面上**（固定模长，消除绝对强度）。语义信息主要编码在**方向**里，而不是模长里——所以 LayerNorm 消除模长的同时保留了语义。

#### 验证：NLP 场景下 BatchNorm 的问题

以 NLP 输入 (batch=2, seq_len=3, embed=4) 为例：

---

In [ ]:
# 句子A：[1,2,3,4], [5,6,7,8], [9,10,11,12]
# 句子B：[10,10,10,10], [20,20,20,20], [30,30,30,30]

# LayerNorm 后：句子A token[0] = [-1.34, -0.45, 0.45, 1.34]（递增关系保留）
#              句子B token[0] = [-1.34, -1.34, -1.34, -1.34]（各维度相同，LayerNorm后仍相同）

# BatchNorm 后（embed 维度跨 batch 归一化）：
# 句子A token[0] 和 句子B token[0] → 相同值！
# 两个语义完全不同的句子被强制拉平了

---

**结论**：BatchNorm 对 NLP 是有害的，因为它破坏了不同 token 之间的语义差异。

#### 现代大模型为什么不用 BatchNorm？

1. **分布式训练的 batch 统计量不稳定**：多 GPU 分割 batch 后，各 GPU 看到的统计量差异大
2. **序列位置无结构**：前面论证过
3. **训练/推理不一致（TID，Training Inference Discrepancy）**：NLP 任务中 batch 统计量和 running statistics 差异大，导致 BN 在 NLP 表现差
4. **主流架构 Pre-LN 的归一化位置不适用 BN**

现代大模型全部使用 **LayerNorm** 或 **RMSNorm**（只缩放不偏移，计算更快）。

---

## 3-5 常见网络层（Linear / Conv2d / LSTM / Embedding）

### nn.Linear — 全连接层

---

In [ ]:
import torch
import torch.nn as nn

# in_features: 输入特征维度
# out_features: 输出特征维度
# bias: 是否有偏置（默认 True）
linear = nn.Linear(10, 5)  # input: (..., 10) → output: (..., 5)

x = torch.randn(2, 10)     # batch=2, 每个样本10维
y = linear(x)              # (2, 5)
print(y.shape)             # torch.Size([2, 5])

# 验证：y = x @ W^T + b
manual = x @ linear.weight.T + linear.bias
print("手动计算一致:", torch.allclose(y, manual))  # True

---

**参数数量**：`weight: (5, 10)` + `bias: (5,)` = 55 个参数

### nn.Conv2d — 卷积层

---

In [ ]:
import torch
import torch.nn as nn

# in_channels: 输入通道数（灰度图=1，RGB=3）
# out_channels: 输出通道数（卷积核数量）
# kernel_size: 卷积核大小
# stride: 步长（默认1）
# padding: 填充（默认0，same 填充需要 padding=kernel//2）

conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)

x = torch.randn(1, 3, 32, 32)  # (batch=1, C=3, H=32, W=32)
y = conv(x)                      # (1, 16, 32, 32)
print(y.shape)

# 验证输出尺寸公式
# H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
h_out = (32 + 2*1 - 3) // 1 + 1
print("计算 H_out:", h_out)  # 32 ✓

---

**参数数量计算**：`kernel × kernel × in_channels × out_channels + out_channels`
= `3 × 3 × 3 × 16 + 16` = `432 + 16` = 448 个参数

### 池化层

---

In [ ]:
# MaxPool2d：取最大值
mp = nn.MaxPool2d(kernel_size=2, stride=2)  # 尺寸减半
# AdaptiveAvgPool2d：输出固定尺寸，自适应核大小
ap = nn.AdaptiveAvgPool2d(output_size=(1, 1))  # 输出 (N, C, 1, 1)
# Global Average Pooling: 常见技巧，用自适应池化实现
gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))  # 把每个通道压成 1 个值

---

### nn.LSTM — 循环层

---

In [ ]:
import torch
import torch.nn as nn

# input_size: 每个时间步的输入特征维度
# hidden_size: 隐藏状态的特征维度
# num_layers: LSTM 层数（堆叠）
# batch_first: True → input shape: (batch, seq, feature)

lstm = nn.LSTM(input_size=10, hidden_size=32, num_layers=2, batch_first=True)

x = torch.randn(4, 8, 10)  # (batch=4, seq_len=8, input_size=10)
h0 = torch.zeros(2, 4, 32)  # (num_layers, batch, hidden_size)
c0 = torch.zeros(2, 4, 32)  # (num_layers, batch, hidden_size)

output, (hn, cn) = lstm(x, (h0, c0))
print("output:", output.shape)    # (4, 8, 32) — 每个时间步的输出
print("hn:", hn.shape)           # (2, 4, 32)  — 最后一个时间步的隐藏状态
print("cn:", cn.shape)           # (2, 4, 32)  — 最后一个时间步的细胞状态

# 如果只想要最后一个时间步的输出（通常用这个）
last_output = output[:, -1, :]  # (4, 32)
print("last:", last_output.shape)

---

### nn.Embedding — 词嵌入层

---

In [ ]:
import torch
import torch.nn as nn

# num_embeddings: 词表大小（最大索引 + 1）
# embedding_dim: 每个词的向量维度
emb = nn.Embedding(num_embeddings=10000, embedding_dim=128)

# 输入：整数索引（LongTensor），范围 [0, num_embeddings)
x = torch.tensor([123, 456, 789])  # batch=3 的句子（每个词是一个索引）
vec = emb(x)                         # (3, 128) — 查表得到三个词的向量
print(vec.shape)                      # torch.Size([3, 128])

# 预训练词向量加载
pretrained = nn.Embedding.from_pretrained(torch.randn(10000, 128))
pretrained.weight.requires_grad = False  # 冻结，只fine-tune顶层

---

#### Embedding 前向传播：数学等价 vs 工程实现

Embedding 在数学上等价于 one-hot 向量与 embedding 矩阵的矩阵乘法，但工程实现是直接内存索引，两者路径不同但结果相同：

| 视角 | 实现 | 路径 |
|------|------|------|
| **数学上** | `one_hot(token_id) @ embedding_table` | 构造稀疏向量 → 矩阵乘 → 得到向量 |
| **工程上** | `embedding_table[token_id]` | 直接地址寻址，无矩阵运算 |

工程上就是 `array[index]` 的直接寻址，和哈希表查表没有本质区别。数学上写成矩阵乘法是为了理论推导方便。

#### Embedding 反向传播：梯度怎么传

核心逻辑：**谁用过我，谁把梯度传给我**。

---

In [ ]:
import torch
import torch.nn as nn

emb = nn.Embedding(num_embeddings=10000, embedding_dim=128)
optimizer = torch.optim.SGD(emb.parameters(), lr=0.01)

# 模拟前向：token_id=42 被使用了两次
input_ids = torch.tensor([42, 7, 42, 99])

# 前向
vec = emb(input_ids)  # shape: (4, 128)
loss = vec.sum()       # 简单 loss

# 反向
loss.backward()

# 验证：token_id=42 被使用了两次，梯度应该累加
print("emb.weight.grad[42]:", emb.weight.grad[42])  # 非零梯度
print("emb.weight.grad[7]:", emb.weight.grad[7])    # 非零梯度
print("emb.weight.grad[99]:", emb.weight.grad[99])  # 非零梯度
print("emb.weight.grad[0]（未使用）:", emb.weight.grad[0])  # 接近零

---

**关键行为**：
- 同一个 token 被多次查询（重复词）→ **梯度累加**
- 没被查过的 token → 梯度为 0，不参与更新
- 计算量正比于 token 数（batch × seq_len），不依赖词表大小

#### 预训练词向量加载与微调策略

实际工程中几乎不会从零训练 embedding，而是加载预训练向量：

---

In [ ]:
import torch
import torch.nn as nn

# 方式一：from_pretrained 加载
pretrained_vectors = torch.randn(10000, 128)  # 实际从 glove/word2vec 加载
emb = nn.Embedding.from_pretrained(pretrained_vectors, freeze=False)
# freeze=False: 训练时继续更新向量（fine-tune）
# freeze=True:  训练时冻结，当静态特征用

---

**四种微调策略**：

| 策略 | 做法 | 适用场景 |
|------|------|---------|
| **Frozen** | `weight.requires_grad = False`，不更新 | 数据量小、领域差异大 |
| **Full Fine-tune** | 全部可训练，正常更新 | 数据量大、领域相近 |
| **Gradual Unfreezing** | 先冻住 → 逐步解冻 | 中等数据量，避免灾难性遗忘 |
| **Adapter** | 冻住主模型，附加小型 MLP 适配器 | 算力有限、保留主模型能力 |

---

In [ ]:
# Gradual Unfreezing 示例：先只训练 head，逐步解冻
model = torchvision.models.resnet18(pretrained=True)

# 阶段1：只训练分类头
for param in model.parameters():
    param.requires_grad = False
model.fc.weight.requires_grad = True

# 训练若干 epoch 后...
# 阶段2：解冻最后两层
for param in model.layer4.parameters():
    param.requires_grad = True

# 阶段3：更多层...
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3  # 解冻后用较小学习率
)

---

---

### Winograd 卷积优化算法

Winograd 算法是深度学习中常用的卷积计算优化技术，核心思想是将卷积中的乘法次数减少。

#### 一维 Winograd F(m, r)：矩阵乘视角

以 F(2, 3) 为例：输入 4 点 `d = [d0, d1, d2, d3]`，滤波器 3 点 `g = [g0, g1, g2]`，输出 2 点：

**直接卷积**（6 次乘）：
```
y0 = d0·g0 + d1·g1 + d2·g2
y1 = d1·g0 + d2·g1 + d3·g2
```

**Winograd 重写形式**：输入组织成 2×3 数据矩阵，滤波器作为 3×1 向量：
```
D = [d0, d1, d2; d1, d2, d3]   (2×3)
y = D · g                        (矩阵乘)
```
这只是一个框架，Winograd 的精妙之处在于：**数据 D 和滤波器 g 都做预变换**，使得主要乘法变成元素乘。

**Winograd 完整公式**：
```
y = A^T · (G·g ⊙ B^T·d)
```

| 步骤 | 操作 | 计算类型 |
|------|------|---------|
| `B^T·d` | 输入变换（4 → 4 点） | 只有加减，无乘 |
| `G·g` | 滤波器变换（3 → 3 点） | 只有加减，无乘 |
| `⊙` | 逐元素乘（4 次） | **只有乘** |
| `A^T` | 输出组合（4 → 2 点） | 只有加减，无乘 |

**总乘法数**：从 6 次降到 **4 次**，代价是十几步加减法。在硬件上乘比加贵得多，所以合算。

#### Winograd F(m, r) 一般形式

Winograd 不是只能 F(2,3)，而是有很多组 (m, r) 可选：

| 形式 | 输出 m | 滤波器 r | 输入点数 | 直接乘法 | Winograd 乘法 |
|------|--------|---------|---------|---------|--------------|
| F(2, 3) | 2 | 3 | 4 | 6 | **4** |
| F(4, 3) | 4 | 3 | 6 | 12 | **6** |
| F(6, 3) | 6 | 3 | 8 | 18 | **8** |
| F(2, 5) | 2 | 5 | 6 | 10 | **6** |

**选大 m 的权衡**：m 越大乘法节省比例越高，但变换矩阵复杂度增加、数值稳定性变差。实际深度学习中选择 F(4,3) 或 F(16,3) 等。

#### 二维 Winograd F(2×2, 3×3)：Khatri-Rao 视角

对于 4×4 输入和 3×3 滤波器，可以分块成 2×3 的 tile，每个 tile 做 2D 卷积：

- 输入 4×4 → 分成重叠的 2×3 块（每个块覆盖一个 2×2 输出区域）
- 每个块是 **2×3 小矩阵**，滤波器是 **3×3 小矩阵**
- 2D 卷积等效为：每个 2×3 块与 3×3 滤波器的矩阵乘

这就是 **Khatri-Rao 乘积**（列对列的逐元素乘）的形式推广。二维 Winograd 的核心洞察：**空间分块 + 滤波器 Khatri-Rao 展开**，使得主要运算变成逐元素乘。

#### 输入不能整除 tile 怎么办

三种处理方式：

**1. 补零（Zero Padding）**：不够的地方补 0，继续用标准 tile，最常用

**2. 剩余处理（Guard / Tail）**：先用标准 tile 覆盖主体，剩下几个点单独用直接卷积（量小，浪费一点乘法无所谓）

**3. 动态选择 tile 大小**：输入长度不是 m 的倍数时，选不规则的最后 tile

#### Winograd 在深度学习中的地位

Winograd 2015-2016 年被工程化后，广泛用于 CNN 推理优化（如 TensorFlow、TensorRT）。但近几年，随着 Tensor Core 等专用矩阵乘硬件的普及，Winograd 在大 tile 场景的优势被削弱——硬件直接做矩阵乘已经足够快。但在**小滤波器（3×3）、中等 batch** 的场景下，Winograd 仍然是重要的优化手段。

---

### 卷积的工程实现：从 for 循环到 im2col 到 cuDNN

#### 为什么 for 循环做卷积极慢

用 for 循环实现单次卷积：

---

In [ ]:
# 最 naive 的实现（假设 stride=1, padding=0）
for oh in range(out_h):
    for ow in range(out_w):
        for kh in range(kernel_h):
            for kw in range(kernel_w):
                val += input[oh+kh, ow+kw] * kernel[kh, kw]
        output[oh, ow] = val

---

四层嵌套 + 大量内存访问 → 极慢，无法利用并行。

#### im2col：把卷积变成矩阵乘法

**核心思想**：把输入的每个滑动窗口"展开"成列，把滤波器展开成行，然后做一次矩阵乘。

```
输入 (H×W) + 滤波器 (K×K)
↓ im2col 展开
矩阵 A (out_h×out_w, K×K)  ×  矩阵 B (K×K, out_channels)
↓                            ↓
每个滑动窗口拉成一列          每个滤波器拉成一行
输出 (out_h×out_w, out_channels)
```

以 4×4 输入、3×3 滤波器、stride=1 为例：
```
输入:
  [d00,d01,d02,d03]
  [d10,d11,d12,d13]
  [d20,d21,d22,d23]
  [d30,d31,d32,d33]

im2col 展开后矩阵 A (4×9):
  [d00,d01,d02,d10,d11,d12,d20,d21,d22]   ← 第1个输出位置的窗口
  [d01,d02,d03,d11,d12,d13,d21,d22,d23]   ← 第2个位置的窗口
  [d10,d11,d12,d20,d21,d22,d30,d31,d32]
  [d11,d12,d13,d21,d22,d23,d31,d32,d33]

滤波器展开成矩阵 B (9×1):
  [k00]
  [k01]
  [k02]
  [k10]
  [k11]
  [k12]
  [k20]
  [k21]
  [k22]

输出 = A @ B  →  矩阵乘!
```

**为什么 im2col 快**：调用高度优化的 BLAS（CPU 上 MKL/OpenBLAS，GPU 上 cuBLAS）矩阵乘，并行度极高。

**内存代价**：im2col 展开后的矩阵 A 有冗余（重叠窗口的数据被重复复制）。但换来的矩阵乘速度提升远大于内存开销。

#### cuDNN：NVIDIA 的卷积底座

cuDNN 是 NVIDIA 提供的深度学习基础库，PyTorch/TensorFlow 等框架的卷积底层都调用它。

**cuDNN 选择算法的方式**：
cuDNN 内部维护多种卷积算法（GEMM-based im2col、Winograd、FFTW 等），每次运行时会对输入 shape 做一次"autotune"——在候选算法上跑一个小算例，选最快的那个缓存下来。

---

In [ ]:
# PyTorch 调用 cuDNN 的示意路径
conv = nn.Conv2d(3, 64, 3)
x = torch.randn(1, 3, 224, 224)

# 实际执行路径：
# 1. PyTorch 解析 conv 参数，构造 cudnnConvolutionDescriptor
# 2. cuDNN 根据 shape 调用 cudnnGetConvolutionForwardAlgorithm (autotune)
# 3. cuDNN 执行选中的算法（im2col + cuBLAS / Winograd / FFTW）
# 4. 结果返回 PyTorch tensor

---

**cuDNN 7 种卷积算法**（按场景选用）：
| 算法 | 适用场景 |
|------|---------|
| GEMM (im2col) | 通用，k×k 大滤波器、large batch |
| Winograd F(2×2, 3×3) | 小滤波器 (3×3)，中等 batch |
| Winograd F(4×4, 3×3) | 较大 tile，batch 大时更明显 |
| FFT | 极大滤波器 (5×5, 7×7)，但内存开销大 |
| Implicit GEMM | 融合了 im2col，不需显式展开，内存更省 |

**cuDNN vs cuBLAS**：cuBLAS 是通用矩阵乘，cuDNN 在其基础上做了 im2col 打包 + filter 打包，然后调 cuBLAS。cuDNN = cuBLAS + im2col + 算法选择 + 反向传播支持。

#### 三种卷积实现的对比

| 实现方式 | 并行度 | 适用场景 | 内存开销 |
|---------|--------|---------|---------|
| for 循环 | 极低 | 教学/验证 | 低 |
| im2col + cuBLAS | 高（矩阵乘） | CPU / 通用 GPU | 中（需展开） |
| cuDNN (autotune) | 最高 | 实际生产 | 可控 |

实际项目中，永远用 cuDNN，PyTorch 默认就调用它，不需要手动指定。

---

### 池化层详解：MaxPool / AvgPool / AdaptiveAvgPool

#### nn.MaxPool2d — 最大池化

---

In [ ]:
import torch.nn as nn

# kernel_size: 池化核大小
# stride: 步长（默认等于 kernel_size）
# padding: 边缘填充
# dilation: 核内空洞间距（膨胀系数）
# ceil_mode: True=向上取整，False=向下取整（默认）

pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1)
x = torch.randn(1, 3, 32, 32)
y = pool(x)  # (1, 3, 16, 16)

---

**前向过程**：在每个 kernel 窗口内取最大值，同时记录最大值位置（indices）。

**反向过程（关键）**：MaxPool 的反向传播**只沿最大值位置传递梯度**，其他位置梯度为 0。这叫 **Max Pooling 的梯度路由**。

---

In [ ]:
# 验证 MaxPool 反向传播
import torch

pool = nn.MaxPool2d(2, stride=2)
x = torch.tensor([[[[1., 2.], [3., 4.]]]], requires_grad=True)
y = pool(x)
loss = y.sum()
loss.backward()

print("x.grad:")  # 只有最大值位置(=4)的梯度为1，其他为0
print(x.grad)     # [[[[0., 0.], [0., 1.]]]]

---

**dilation（膨胀/空洞）参数**：

---

In [ ]:
pool_dilated = nn.MaxPool2d(kernel_size=3, dilation=2)
# 等效感受野 = kernel_size + (kernel_size-1)*(dilation-1) = 3 + 2*1 = 5
# 但实际 kernel 还是 3×3，参数不变，只是采样时跳过了空洞

---

#### nn.AvgPool2d — 平均池化

---

In [ ]:
avg = nn.AvgPool2d(kernel_size=2, stride=2)
# 前向：取 kernel 内均值
# 反向：梯度平均分配给 kernel 内所有位置（= 1/(k*k)）

---

#### Global Average Pooling (GAP)

---

In [ ]:
gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))
x = torch.randn(4, 512, 7, 7)
y = gap(x)  # (4, 512, 1, 1) → squeeze → (4, 512)

---

GAP 将每个通道压缩为 1 个值，完全替代 FC 层。优点：**无参数，不易过拟合**，Inception/ResNet 等现代网络最后常用 GAP。

#### AdaptiveAvgPool2d — 自适应输出尺寸

---

In [ ]:
# 自适应输出到指定尺寸，框架自动计算 kernel_size 和 stride
adaptive = nn.AdaptiveAvgPool2d(output_size=(3, 3))  # 输出 3×3
adaptive = nn.AdaptiveAvgPool2d(output_size=(1, 1))   # 输出 1×1（GAP）

---

无论输入是 7×7 还是 14×14，自适应池化都能输出 3×3，**不需要关心输入尺寸**。

---

### nn.BatchNorm2d — 批归一化深入

#### 训练 vs 推理的统计量差异

---

In [ ]:
bn = nn.BatchNorm2d(num_features=64)

bn.train()   # 训练模式
bn.eval()    # 推理模式

---

| 模式 | 均值/方差来源 | `running_mean/var` |
|------|------------|-------------------|
| train | 当前 batch 实时计算 | **会更新**（指数移动平均） |
| eval | 不计算，用全局统计量 | **不变**，训练时累积的值 |

**running_mean/var 的更新公式**：
```
running_mean = (1 - momentum) * running_mean + momentum * batch_mean
running_var  = (1 - momentum) * running_var  + momentum * batch_var
```

momentum 默认 0.1（PyTorch 中 `momentum=0.1`），意义：**快速追踪近期 batch 统计量变化**。

#### track_running_stats 参数

---

In [ ]:
bn = nn.BatchNorm2d(64, track_running_stats=False)
# 推理时也用 batch 统计量（用于测试集和训练集分布差异大的场景）

---

#### 推理时 BN 的确定性行为

eval 模式下，`BatchNorm` 的行为是**完全确定性**的：

---

In [ ]:
bn.eval()
y1 = bn(x1)  # 无论 x1 是什么，running_mean/var 都固定
y2 = bn(x2)  # y1 和 y2 不可能相等，因为 x 不同 → 但 running_stats 确实不变

---

**关键误解澄清**：eval 模式不是"用 running_mean 代替 batch_mean"，而是：
- **训练阶段**：归一化用 batch 统计量（实时计算）
- **推理阶段**：归一化用 running 统计量（EMA 累积）

两者公式完全一样，只是"均值/方差从哪来"不同。

#### BatchNorm 在什么时候更新 running statistics

---

In [ ]:
bn.train()
bn.eval()  # ← 切换到 eval 后，running_mean/var **完全停止更新**

bn.eval()
bn.train()  # ← 切换回 train 后，继续用 batch 统计量更新 running_mean/var

---

---

### nn.Dropout — 随机失活深入

#### 训练模式的 mask 生成机制

---

In [ ]:
dropout = nn.Dropout(p=0.5)  # p=丢弃概率

x = torch.tensor([1., 2., 3., 4., 5.])
y_train = dropout(x)
# 前向：随机生成 0/1 mask，以 p 概率置 0，剩余 / (1-p)
# 反向：mask=0 的位置梯度=0（不更新），mask=1 的位置正常回传

---

**mask 生成过程（PyTorch 内部）**：

---

In [ ]:
# 伪代码
mask = (torch.rand(x.shape) > p) / (1 - p)  # 注意除以 (1-p)
y = x * mask

---

#### 反向传播：mask 本身不参与梯度计算

Dropout 反向传播的梯度 = 上游梯度 × mask，关键：**mask 在反向时是固定的**（和前向用同一个 mask），mask 本身不产生梯度。

---

In [ ]:
# 验证：Dropout 反向传播
x = torch.tensor([1., 2., 3.], requires_grad=True)
dropout = nn.Dropout(p=0.5)
y = dropout(x)
loss = y.sum()
loss.backward()

# x.grad: 某些位置梯度为 0（被 mask 丢弃），某些位置正常
# x.grad != 0 的位置和前向 mask=1 的位置完全一致

---

#### Inverted Dropout（PyTorch 使用的方式）

训练时做缩放（除以 1-p），推理时不做任何操作：

---

In [ ]:
# Inverted Dropout（PyTorch / TF / 大多数框架）
y = x * mask / (1-p)  # 训练时

# 推理时：所有神经元参与，输出期望 E[y] = x

---

优势：**推理代码和训练代码完全一致**，只需切换 mode。相对于"推理时手动缩放"更简洁。

---

### nn.ConvTranspose2d — 转置卷积（反卷积）

转置卷积不是卷积的逆运算，而是**卷积的梯度运算**（conv transpose = gradient with respect to input）。

---

In [ ]:
conv = nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=3, stride=2, padding=1, output_padding=1)
# stride=2: 输出尺寸 = 输入尺寸 × 2（upsample）
# output_padding=1: 消除 stride>1 时的一个单元偏移

---

**用途**：上采样（GAN、U-Net、语义分割）、生成器网络。

**输出尺寸公式**：
```
H_out = (H_in - 1) * stride - 2*padding + dilation*(kernel_size-1) + output_padding + 1
```

---

In [ ]:
# 示例
deconv = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, padding=1, output_padding=1)
x = torch.randn(1, 1, 4, 4)
y = deconv(x)  # (1, 1, 8, 8) — 4×2=8

---

---

### nn.LSTM — 长短期记忆网络深入

#### LSTM 的四个门控机制

---

In [ ]:
import torch.nn as nn

lstm = nn.LSTM(input_size=10, hidden_size=32, num_layers=2, batch_first=True, bidirectional=False)

x = torch.randn(4, 8, 10)  # (batch, seq_len, input_size)
output, (hn, cn) = lstm(x)

---

LSTM 内部有 4 个门控，计算公式：

```
f_t = σ(W_f · [h_{t-1}, x_t] + b_f)     # 遗忘门：决定丢弃多少旧信息
i_t = σ(W_i · [h_{t-1}, x_t] + b_i)     # 输入门：决定写入多少新信息
C'_t = tanh(W_C · [h_{t-1}, x_t] + b_C) # 候选记忆：新的候选值
C_t = f_t * C_{t-1} + i_t * C'_t         # 细胞状态：遗忘+输入的组合

o_t = σ(W_o · [h_{t-1}, x_t] + b_o)     # 输出门：决定输出多少
h_t = o_t * tanh(C_t)                    # 隐藏状态：输出门 × tanh(细胞状态)
```

#### 为什么 LSTM 能避免梯度消失

关键在于**细胞状态 C_t 的更新方式**：

```
C_t = f_t * C_{t-1} + i_t * C'_t
```

- 遗忘门 f_t 接近 1 时，梯度可以几乎无损地传回很久以前的时间步
- 加法操作（而非连乘）是 LSTM 避免梯度消失的核心：**∂C_t/∂C_{t-1} = f_t**，不是连乘
- 梯度沿细胞状态路径传递时，"*"操作被"+"替代，梯度传播变成加法而非乘法

#### packed_padded_sequence 与 padding mask

处理变长序列时，需要 pad 后一起 batch，但 pad 的位置不应参与计算：

---

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 变长序列
lengths = [5, 3, 7]  # 各序列实际长度
x_padded = torch.nn.utils.rnn.pad_sequence([...], batch_first=True)

# 打包（按长度降序，自动处理 pad）
packed = pack_padded_sequence(x_padded, lengths, batch_first=True, enforce_sorted=True)
output_packed, (hn, cn) = lstm(packed)

# 解包
output, _ = pad_packed_sequence(output_packed, batch_first=True)
# output[:, 3, :] 在 lengths[0]=5 时，位置3的数据是真实的，位置>3是 pad 的

---

---

### nn.Softmax / LogSoftmax — 归一化指数函数

---

In [ ]:
import torch.nn as nn

sm = nn.Softmax(dim=-1)
log_sm = nn.LogSoftmax(dim=-1)  # 用于 CrossEntropyLoss（数值稳定）

x = torch.tensor([1., 2., 3.])
y = sm(x)  # [e^1, e^2, e^3] / (e^1+e^2+e^3) ≈ [0.09, 0.24, 0.67]
y_log = log_sm(x)  # log([...]) ≈ [-2.41, -1.41, -0.41]

---

**dim 很重要**：在哪个维度做 softmax，就在哪个维度做归一化（和为 1）。

---

In [ ]:
# (batch, seq_len, vocab_size) = (2, 5, 10000)
logits = torch.randn(2, 5, 10000)
sm = nn.Softmax(dim=-1)  # ← 在 vocab_size 维度归一化 → 每词概率分布
attn_weights = sm(logits)  # 每个位置一个概率分布

---

---

### 激活函数对比总结

| 激活函数 | 公式 | 输出范围 | 优点 | 缺点 |
|---------|------|---------|------|------|
| **ReLU** | max(0, x) | [0, +∞) | 计算快、无梯度饱和 | 神经元死亡问题 |
| **LeakyReLU** | x if x>0 else α·x | (-∞, +∞) | 避免神经元死亡 | 多了超参 α |
| **GELU** | x·Φ(x) | (-∞, +∞) | Transformer 最常用，平滑 | 计算略慢 |
| **SiLU / Swish** | x·sigmoid(x) | (-∞, +∞) | 自门控，比 ReLU 更平滑 | 计算慢 |
| **Sigmoid** | 1/(1+e^{-x}) | (0, 1) | 概率输出 | 梯度易饱和、梯度消失 |
| **Tanh** | (e^x - e^{-x})/(e^x + e^{-x}) | (-1, 1) | 零中心 | 梯度易饱和 |

**工程选择**：
- CNN / 通用 → **ReLU**（最快）
- Transformer / Attention → **GELU**（BERT、ViT、GPT 等）
- 需要概率输出 → **Sigmoid**（多标签分类）
- 非线性输出 → **Tanh**（LSTM 门控内部常用）

**GELU vs ReLU**：GELU 是 ReLU 的平滑近似，梯度在负区间也有小幅非零值，不会完全"死亡"。Transformer 时代 GELU 成为默认选择。

---

## 3-6 Sequential 与模块化设计

### nn.Sequential — 两种定义方式

---

In [ ]:
import torch
import torch.nn as nn

# 方式一：直接传入（位置索引，调试不方便）
model = nn.Sequential(
    nn.Linear(10, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 2)
)

---

In [ ]:
# 方式二：OrderedDict 命名（推荐，调试友好）
from collections import OrderedDict

model = nn.Sequential(OrderedDict([
    ('fc1',   nn.Linear(10, 8)),
    ('relu1', nn.ReLU()),
    ('fc2',   nn.Linear(8, 4)),
    ('relu2', nn.ReLU()),
    ('fc3',   nn.Linear(4, 2))
]))

---

### Sequential 内部如何工作

Sequential 本身就是一个 `nn.Module`，`forward` 按顺序执行每一层：

---

In [ ]:
# Sequential 内部 forward 等价于：
def forward(self, x):
    for layer in self._modules.values():
        x = layer(x)
    return x

---

**OrderedDict vs 直接传入的区别**：两者功能完全相同，OrderedDict 只是给每一层起了名字，使得 `named_children()` 和 `state_dict()` 的 key 更清晰。

---

In [ ]:
for name, module in model.named_children():
    print(name, '→', module)

# OrderedDict 输出：
# fc1 → Linear(in=10, out=8)
# relu1 → ReLU()
# fc2 → Linear(in=8, out=4)
# relu2 → ReLU()
# fc3 → Linear(in=4, out=2)

---

### Sequential 的局限：四种必须自定义的场景

Sequential 只能**线性串联**，以下四种情况必须自定义 Module：

**1. 残差连接**

---

In [ ]:
# Sequential 实现不了，必须自定义
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):
        return self.net(x) + x  # ← 跳跃连接

---

**2. 多输入**

---

In [ ]:
# 多输入必须自定义
class TextImageFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_net = nn.Linear(768, 256)
        self.img_net  = nn.Linear(512, 256)

    def forward(self, text, img):
        return self.text_net(text) * self.img_net(img)  # ← 多输入

---

**3. 多输出**

---

In [ ]:
# 多输出必须自定义
class DualOutputNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Linear(512, 10)
        self.embedding  = nn.Linear(512, 128)

    def forward(self, x):
        return {
            'logits': self.classifier(x),
            'features': self.embedding(x)
        }

---

**4. 共享层**

---

In [ ]:
# 同一层在两处使用，必须自定义
class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Linear(512, 256)  # ← 两处共享

    def forward(self, x1, x2):
        return self.shared(x1), self.shared(x2)  # ← 同一层走两次

---

### Sequential 适用场景对照表

| 场景 | Sequential | 自定义 Module |
|------|-----------|--------------|
| 线性串联：Conv → BN → ReLU → Pool | ✅ | ✅ |
| 快速原型：几行搭一个 MLP | ✅ | ✅ |
| 残差连接（ResNet Block） | ❌ | ✅ |
| 多分支网络（FPN、Inception、U-Net） | ❌ | ✅ |
| 多输入/多输出（多模态） | ❌ | ✅ |
| 共享层（Siamese Network） | ❌ | ✅ |

### 模型设计的分层思维

```
Input → [Conv → BN → ReLU → Pool] × N → FC → Output
              ↑
         用 nn.Sequential 打包
```

---

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        return self.net(x)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = ConvBlock(3, 32)
        self.block2 = ConvBlock(32, 64)
        self.block3 = ConvBlock(64, 128)
        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = x.mean(dim=[2, 3])  # Global Average Pooling
        return self.fc(x)

---

### 实际工程选择原则

- **简单任务 / 原型验证** → Sequential 够用，几行代码搞定
- **生产级 / 复杂结构** → 自定义 Module（更清晰可控，可读性好）
- **永远不确定** → 自定义 Module，因为后续改起来容易

---

## 面试常考点

**Q1: `nn.Module` 的 `forward` 为什么只需写前向，反向自动搞定？**
- PyTorch autograd 根据前向运算自动构建计算图，每种运算对应一个反向函数（grad_fn）
- `model(x)` 实际调用 `nn.Module.__call__(x)` → 执行 forward → 执行 hooks

**Q2: `model(img)` 背后发生了什么？**
- `nn.Module.__call__` 被调用 → 执行 `forward()` → 执行所有注册的 `forward_hooks` → 返回输出
- `__call__` 和直接调 `forward()` 的区别在于 hooks 是否被执行

**Q3: `model.train()` vs `model.eval()` 区别？**
- `train()`：BatchNorm 用当前 batch 的均值/方差，Dropout 随机置零
- `eval()`：BatchNorm 用全局 moving average，Dropout 全部通过
- 两者都只影响特定层的统计行为，不影响其他层

**Q4: 如何冻结部分层？冻结后梯度怎么传？**

---

In [ ]:
for param in model.layer1.parameters():
    param.requires_grad = False
optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],  # optimizer 只更新解冻的参数
    lr=0.01
)

---

- 冻结（`requires_grad=False`）：梯度**原封不动**往上继续传，只是该参数本身不记录梯度
- 对比 `detach()`：真正截断梯度流，梯度停止传播
- 冻结不等同于"置零"或"常数"，它仍然参与计算图，只是可学习性被关闭

**Q5: `state_dict()` 保存的是什么？**
- 所有 `nn.Parameter` 的字典，key 是参数名（如 `layer1.weight`），value 是张量
- 不包含模型结构（需要先实例化再 load）

**Q6: `children()` vs `modules()` 区别？**
- `children()`：只返回直接子模块（深度1层）
- `modules()`：返回所有模块，包括嵌套的子子模块（深度遍历）

# 四、torch.optim — 优化器

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| SGD + Momentum | 基础优化器 + 动量加速 | momentum 是什么，为什么能加速 |
| RMSprop / Adagrad | 自适应学习率早期方案 | 解决什么问题 |
| Adam | 自适应学习率 + 偏置校正 | 一阶/二阶矩估计是什么 |
| AdamW | Adam + weight decay 分离 | 为什么比 Adam + L2 好 |
| 学习率调度 | `lr_scheduler` | 常用调度策略及场景 |
| 参数分组 | 不同层不同学习率 | 怎么配 |
| Gradient Clipping | 梯度裁剪 | 防止梯度爆炸 |
| Zero_grad + 状态保存 | 清零梯度 + 保存加载 | 为什么要手动调用 |

**⚠️ 面试必答题：**
- SGD 和 Adam 的区别？各自适用场景？
- AdamW 和 Adam + L2 正则的区别？
- 学习率衰减策略有哪些？
- 为什么梯度要用 `zero_grad()` 清零，不能累加？
- 梯度裁剪具体怎么操作？

---

## 4-1 SGD 与 Momentum

### SGD（随机梯度下降）

**核心**：每一步独立更新，看当前梯度，不依赖历史。

In [ ]:
Δw = lr × gradient
w = w - Δw

**特点**：
- 稳定，但收敛慢（特别是缓坡）
- 学习率调不好容易震荡

### Momentum（动量）

**核心**：历史梯度的指数加权累积，再用来更新参数。

**公式：**

In [ ]:
momentum = γ × momentum + lr × gradient
w = w - momentum

**展开形式（关键）：**
```
momentum_t = lr × [g_t + γ·g_{t-1} + γ²·g_{t-2} + γ³·g_{t-3} + ...]
```

- γ=0.9 时，10步前的梯度权重只剩约 35%
- 所以 momentum ≈ 过去 ~10 步梯度的加权平均 × lr
- **本质：参数更新量 = lr × 历史梯度的指数加权累积**

**直观例子**（lr=0.1, γ=0.9, momentum=0）：

| Step | 当前梯度 | Momentum | 参数更新 Δw | 累计更新 |
|------|---------|---------|-----------|---------|
| 1 | 3.0 | 0.9×0 + 0.1×3.0 = 0.30 | -0.30 | -0.30 |
| 2 | 2.0 | 0.9×0.30 + 0.1×2.0 = 0.47 | -0.47 | -0.77 |
| 3 | 1.0 | 0.9×0.47 + 0.1×1.0 = 0.52 | -0.52 | -1.29 |
| 4 | 0.5 | 0.9×0.52 + 0.1×0.5 = 0.52 | -0.52 | -1.81 |

**vs 无动量SGD**（同样4步梯度）：累计更新只有 **-0.65**，动量版是 **-1.81**，快了近3倍。

**为什么能加速：**
- 方向一致：速度累积，越走越快
- 方向振荡：正负抵消，自动过滤噪音

**为什么叫"动量"**：物理上像滚球下山，即使坡度变缓，惯性也会推着继续滚。

### Nesterov 动量

**核心**：先按惯性走一步，在那个位置预判梯度，再回来更新。

In [ ]:
preview_w = w - γ × momentum          # 预判位置（先按惯性走）
gradient_preview = 在 preview_w 处算的梯度  # 预判梯度
momentum = γ × momentum + lr × gradient_preview
w = w - momentum

**直观理解**：
- 普通 Momentum：低头冲，撞到墙才减速
- Nesterov：抬头看路，快到墙了提前减速

**代码：**

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, 
                           momentum=0.9, nesterov=True)

**三种方式对比**（lr=0.1, γ=0.9）：

| | 无动量 SGD | 带动量 SGD | Nesterov |
|---|---|---|---|
| 每步更新 | lr × g_t | lr × [g_t + γ·g_{t-1} + ...] | 预判位置算梯度，累积同动量 |
| 速度 | 无 | 有累积 | 有累积 |
| 减速机制 | 无 | 靠当前梯度变小 | **提前**感知梯度变小 |
| 4步累计更新 | -0.65 | -1.81 | 更稳（真实场景优于动量） |

### zero_grad 三种方式

梯度清零有三个入口，语义略有不同：

In [ ]:
optimizer.zero_grad()    # 优化器清零：只清它管理的那些参数的梯度
model.zero_grad()       # 模型清零：递归清所有子模块的梯度
for p in model.parameters():
    p.grad = None       # 手动清零：最直接

**为什么 PyTorch 把 zero_grad 放在 optimizer 上？**
- optimizer 创建时就绑定了要管理的参数
- `optimizer.zero_grad()` = "把管的这批参数的梯度清零"
- 语义自然：谁管谁清

**PyTorch 默认累加梯度**（不是覆盖），方便大 batch：

In [ ]:
loss.backward()      # grad += 梯度
loss.backward()      # grad += 新梯度（累加了！）
optimizer.zero_grad()  # 下一步前必须清零

---

## 4-2 RMSprop / Adagrad

### Adagrad

**特点**：每个参数独立自适应学习率，梯度大的参数学习率衰减快。

**适用**：稀疏特征（如 NLP、Embedding）。

In [ ]:
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01, 
                                lr_decay=0.01, weight_decay=0)

**问题**：学习率会单调下降，后期训练困难。

### RMSprop

**改进**：引入滑动平均，平滑学习率变化。

**公式：**
```
cache = γ * cache + (1-γ) * gradient²
param = param - lr * gradient / (√cache + ε)
```

- γ: 衰减系数，通常 0.99
- ε: 防止除零，通常 1e-8

In [ ]:
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001, 
                               alpha=0.99, eps=1e-8)

---

## 4-3 Adam 原理

### Adam（Adaptive Moment Estimation）

**目前最流行的优化器**，结合了 Momentum 和 RMSprop。

**核心思想**：用一阶矩估计（类似动量）和二阶矩估计（类似 RMSprop的自适应学习率）分别修正梯度。

**公式：**
```
# 一阶矩（动量估计）
m_t = β1 * m_{t-1} + (1-β1) * gradient

# 二阶矩（梯度的方差估计）
v_t = β2 * v_{t-1} + (1-β2) * gradient²

# 偏置校正（至关重要！）
m_hat = m_t / (1 - β1^t)
v_hat = v_t / (1 - β2^t)

# 更新参数
param = param - lr * m_hat / (√v_hat + ε)
```

- β1: 一阶矩衰减，通常 0.9
- β2: 二阶矩衰减，通常 0.999
- ε: 防止除零，通常 1e-8
- t: 迭代次数（从 1 开始）

**为什么需要偏置校正？**
- 初始化时 m_0 = 0, v_0 = 0
- 如果不校正：第一次更新 lr * gradient（实际学习率被缩小）
- 校正后：第一轮就是正确学习率

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, 
                          betas=(0.9, 0.999), eps=1e-8)

### Adam 适用场景

| 推荐使用 | 不推荐 |
|---|---|
| 深度网络（CNN/Transformer） | 简单线性模型 |
| 非稀疏数据 | 需要精确调参的场景 |
| 快速实验 | lr 敏感的任务 |

---

## 4-4 AdamW vs Adam + L2

### 核心区别

**Adam + L2**：
```
grad = gradient + weight_decay * param
m_t = β1 * m_{t-1} + (1-β1) * grad
```

**AdamW**：
```
m_t = β1 * m_{t-1} + (1-β1) * gradient
param = param - lr * (m_hat / √v_hat + weight_decay * param)
```

区别：**L2 正则化通过梯度更新（影响动量估计）**，而 **AdamW 在参数更新时直接减（ weight_decay * param * lr）**。

### 为什么 AdamW 更好

1. **解耦**：学习率和正则化强度独立
2. **理论保证**：与 L2 正则化的原始目标函数等价
3. **实践效果**：通常 weight_decay 设置为 0.01~0.05（而非 Adam+L2 的 1e-4）

In [ ]:
# Adam + L2（传统方式）
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# AdamW（推荐方式）
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

**面试话术**：
> "AdamW 是 Adam 的改进，把 weight_decay 从梯度项移到了参数更新项，避免了动量估计被 L2 正则化污染。实际应用中通常用 0.01-0.05 的 weight_decay，比 Adam+L2 的 1e-4 高两个数量级。"

---

## 4-5 学习率调度（lr_scheduler）

### 常用策略

| 调度器 | 公式 | 适用场景 |
|---|---|---|
| StepLR | 每 N 个 epoch 固定衰减 | 普通训练 |
| MultiStepLR | 指定 epoch 列表衰减 | 已知转折点 |
| CosineAnnealingLR | 余弦曲线 | 图像分类 |
| ReduceLROnPlateau | 指标不降时衰减 | 验证集优化 |
| Warmup + Cosine | 先升后降 | Transformer |

### StepLR

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

for epoch in range(50):
    train()
    scheduler.step()
    print(f"Epoch {epoch}: lr={optimizer.param_groups[0]['lr']}")
# Epoch 0-9: 0.001
# Epoch 10-19: 0.0005
# Epoch 20-29: 0.00025
# ...

### CosineAnnealingLR

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

### Warmup + Cosine（在 PyTorch 2.0+）

In [ ]:
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=0.1,
    epochs=50,
    steps_per_epoch=len(dataloader)
)

###ReduceLROnPlateau

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

for epoch in range(50):
    val_loss = validate()
    scheduler.step(val_loss)  # 根据指标调整

**面试话术**：
> "常用的是 StepLR（简单）、CosineAnnealing（效果好）和 ReduceLROnPlateau（自动）。Transformer 类任务一般用 OneCycleLR 先升后降，能获得更好收敛。"

---

## 4-6 参数分组（不同层不同学习率）

### 场景

- 预训练模型微调：特征提取层用小学习率，分类头用大学习率
- 不同层用不同weight_decay

### 实现

In [ ]:
# 特征提取层（卷积层）：小学习率，无 weight_decay
base_params = [p for n, p in model.named_parameters() if 'feature' in n]
# 分类头：大学习率，有 weight_decay
head_params = [p for n, p in model.named_parameters() if 'head' in n]

optimizer = torch.optim.AdamW([
    {'params': base_params, 'lr': 1e-4, 'weight_decay': 0},
    {'params': head_params, 'lr': 1e-3, 'weight_decay': 0.01}
])

### 另一种写法（通过 dict）

In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.feature.parameters(), 'lr': 1e-4},
    {'params': model.head.parameters(), 'lr': 1e-3}
], weight_decay=0.01)
# 注意：weight_decay 对所有参数生效，可设为 0，再用 params 指定

---

## 4-7 Gradient Clipping（梯度裁剪）

### 为什么要裁剪

防止梯度爆炸（尤其 LSTM、Transformer、GAN）。

**常见阈值**：1.0 或 5.0

### 两种方式

#### nn.utils.clip_grad_norm_
按全体参数 L2 范数裁剪：

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

# 计算方式
total_norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in model.parameters()))
if total_norm > max_norm:
    scale = max_norm / total_norm
    for p in model.parameters():
        p.grad *= scale

#### nn.utils.clip_grad_value_
按单个参数值裁剪：

In [ ]:
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=1.0)
# 所有参数的梯度被裁到 [-1.0, 1.0]

**推荐**：clip_grad_norm_（更常用）

In [ ]:
for inputs, targets in dataloader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = loss_fn(outputs, targets)
    loss.backward()
    
    # 梯度裁剪
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()

**面试话术**：
> "梯度裁剪防止梯度爆炸，尤其是 RNN/Transformer。我在实际项目中一般用 clip_grad_norm_，阈值设为 1.0 或 5.0，效果很好。"

---

## 4-8 优化器状态保存

optimizer 有内部状态（如 Adam 的 momentum、方差估计），保存 checkpoint 时必须一起保存。

In [ ]:
# 保存
checkpoint = {
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'epoch': epoch
}
torch.save(checkpoint, 'checkpoint.pth')

# 加载
checkpoint = torch.load('checkpoint.pth')
model.load_state_dict(checkpoint['model'])
optimizer.load_state_dict(checkpoint['optimizer'])
epoch = checkpoint['epoch']

**注意**：不同 optimizer 状态结构不同，Adam 的 state_dict 不能加载到 SGD。

**set_to_none=True**（更高效的清零方式）:

In [ ]:
optimizer.zero_grad(set_to_none=True)
# 设为 None 而非 0，内存更省

---

## 面试汇总

### Q1: SGD 和 Adam 的区别？

| | SGD | Adam |
|---|---|---|
| 学习率 | 固定（需手动调） | 自适应 |
| 收敛速度 | 慢 | 快 |
| 调参难度 | 高 | 低 |
| 适用场景 | 简单模型、精确调参 | 深度网络 |

### Q2: AdamW 和 Adam + L2 的区别？

- Adam + L2：L2 正则化进入梯度，影响动量估计
- AdamW：weight_decay 在参数更新时直接减，独立于动量
- 推荐用 AdamW

### Q3: 常用学习率衰减策略？

- StepLR：每 N 个 epoch 固定衰减
- CosineAnnealing：余弦曲线
- ReduceLROnPlateau：验证集不降时自动降
- OneCycleLR：先升后降（Transformer 推荐）

### Q4: 为什么 zero_grad() 必须手动调用？

- PyTorch 默认梯度累加（支持大 batch）
- 每次 backward 会累加到 .grad，所以要清零

### Q5: 梯度裁剪怎么做？

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

---

## 代码练习

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. 准备数据
X = torch.randn(1000, 10)
y = torch.randn(1000, 1)
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32)

# 2. 模型
model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 1))

# 3. 优化器：AdamW + 参数分组
optimizer = optim.AdamW([
    {'params': model[0]..parameters(), 'lr': 1e-3},  # 卷积层
    {'params': model[2].parameters(), 'lr': 1e-4}     # 输出层
], weight_decay=0.01)

# 4. 学习率调度
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# 5. 训练循环
for epoch in range(10):
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad(set_to_none=True)
        output = model(batch_x)
        loss = nn.MSELoss()(output, batch_y)
        loss.backward()
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
    scheduler.step()
    print(f"Epoch {epoch}: loss={loss.item():.4f}, lr={optimizer.param_groups[0]['lr']:.6f}")

---

## 参考资料

- PyTorch 官方文档: https://pytorch.org/docs/stable/optim.html
- 论文: Adam: A Method for Stochastic Optimization (2014)
- 论文: Decoupled Weight Decay Regularization (2019) — AdamW

# 五、DataLoader — 数据加载

**核心知识点：**

| 知识点 | 说明 | 面试高频追问 |
|---|---|---|
| `Dataset` | 自定义数据抽象，必须实现 `__getitem__` + `__len__` | 如何自己实现 |
| `DataLoader` | batch / shuffle / num_workers | 各参数含义 |
| `collate_fn` | 自定义 batch 拼接 | 什么时候需要重写 |
| `pin_memory` | 加速 GPU 传输 | 什么原理 |
| `torchvision` | 图像领域数据集（MNIST / CIFAR / ImageNet） | |

**⚠️ 面试必答题：**
- DataLoader 的 shuffle 是在哪个层面做的？
- num_workers 设置过大的副作用是什么？

---

## 5-1 自定义 Dataset

## 5-2 DataLoader 核心参数

## 5-3 collate_fn 与变长序列

## 5-4 pin_memory 与加速

## 5-5 torchvision 内置数据集

---

## 代码练习

# 六、模型保存与加载

| 方式 | 代码 | 适用场景 |
|---|---|---|
| 保存 state_dict | `torch.save(model.state_dict(), path)` | **推荐**，轻量 |
| 保存整个模型 | `torch.save(model, path)` | 不推荐，依赖类定义 |
| 加载 | `model.load_state_dict(torch.load(path))` | 常用方式 |
| 只加载部分参数 | `strict=False` / 过滤 key | 迁移学习/微调 |
| 保存优化器状态 | `torch.save({'model': ..., 'optimizer': ...}, path)` | 断点续训 |

---

## 6-1 state_dict 方式

## 6-2 断点续训（Checkpoint）

## 6-3 部分加载（strict=False）

## 6-4 跨设备保存与加载

---

## 代码练习

# 七、GPU 加速与分布式

| 知识点 | 说明 |
|---|---|
| 单 GPU | `model.cuda()` / `tensor.to(device)` |
| 多 GPU | `nn.DataParallel(model, device_ids=[0,1,2])` |
| 多 GPU 原理 | 按 batch 维度分割 → 各 GPU 独立 forward → 梯度累加到主 GPU |
| 分布式 DDP | `DistributedDataParallel` — 工业级多机多卡 |
| `torch.cuda.is_available()` | 判断 GPU 是否可用 |

---

## 7-1 单 GPU 迁移

## 7-2 DataParallel（DP）多卡

## 7-3 DistributedDataParallel（DDP）原理

## 7-4 多卡实验与验证

---

## 代码练习

# 八、Fine-tuning（微调）

| 方式 | 做法 |
|---|---|
| 局部微调 | 冻结底层参数（`requires_grad=False`），只训练顶层 |
| 全局微调 | 不同层设不同学习率（在 optimizer param_groups 中配置） |
| 加载预训练 | `model = torchvision.models.resnet18(pretrained=True)` |

---

## 8-1 预训练模型加载

## 8-2 冻结层微调（Frozen）

## 8-3 不同层不同学习率

## 8-4 Adapter 简介

---

## 代码练习

# 九、常见网络层速查

```
卷积:  nn.Conv2d(in, out, kernel, stride, padding)
池化:  nn.MaxPool2d / nn.AvgPool2d / nn.AdaptiveAvgPool2d
BN:    nn.BatchNorm2d(channels)  — train时用batch统计，eval时用全局统计
Dropout: nn.Dropout(p)           — train时随机置0，eval时全部保留
全连接: nn.Linear(in_features, out_features)
激活:  nn.ReLU / nn.Sigmoid / nn.Tanh
```

**卷积层参数说明：** in_channels / out_channels / kernel_size / stride / padding

---

## 9-1 卷积层（Conv2d）

## 9-2 池化层（MaxPool / AvgPool / AdaptiveAvgPool）

## 9-3 归一化层（BatchNorm / LayerNorm / InstanceNorm）

## 9-4 Dropout 与正则化

## 9-5 激活函数对比

## 9-6 常用组合

---

## 代码练习

# 十、训练流程全链路（必须能默写）

```
1. 定义 Dataset + DataLoader
2. 定义模型 (继承nn.Module) → 放到GPU
3. 定义损失函数 (CrossEntropyLoss / MSELoss...)
4. 定义优化器 (SGD / Adam)
5. 训练循环:
   for epoch in range(E):
       model.train()
       for batch in train_loader:
           optimizer.zero_grad()
           output = model(input)
           loss = criterion(output, target)
           loss.backward()
           optimizer.step()
       model.eval()
       with torch.no_grad():
           ...
```

---

## 10-1 完整训练循环

## 10-2 验证与测试流程

## 10-3 训练监控（Loss 曲线、Early Stopping）

## 10-4 常见问题与调试

---

## 代码练习